# 1. Corpus Discovery & Inventory

This section performs the first-stage audit of the RAGScholar research corpus.

The goal is to verify that the expected dataset structure exists and to create a reliable inventory of all source files before performing any PDF-level validation.

### Expected corpus structure

```text
data/
├── Textbooks/   → 8 documents
├── Survey/      → 23 documents
└── Landmark/    → 24 documents

In [2]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

EXPECTED = {"Textbooks": 8, "Survey": 23, "Landmark": 24}
PREFIX = {"Textbooks": "T", "Survey": "S", "Landmark": "L"}
EXPECTED_TOTAL = sum(EXPECTED.values())

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if all((p/"data"/c).is_dir() for c in EXPECTED)), None)
assert PROJECT_ROOT is not None, "Project root not found. Expected: data/Textbooks, data/Survey, data/Landmark"

DATA_DIR = PROJECT_ROOT/"data"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("EXPECTED_TOTAL:", EXPECTED_TOTAL)

PROJECT_ROOT: c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt
EXPECTED_TOTAL: 55


## 1.1 Project Folder Validation

In [3]:
folder_df = pd.DataFrame([{"category": c, "path": str(DATA_DIR/c), "exists": (DATA_DIR/c).exists(), "is_directory": (DATA_DIR/c).is_dir()} for c in EXPECTED])

display(folder_df)

folder_status = bool(folder_df["exists"].all() and folder_df["is_directory"].all())
print("STATUS:", "PASS ✅" if folder_status else "FAIL ❌")

,category,path,exists,is_directory
0,Textbooks,c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Textbooks,True,True
1,Survey,c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Survey,True,True
2,Landmark,c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Landmark,True,True


STATUS: PASS ✅


## 1.2 PDF File Enumeration

In [4]:
records = []

for category, prefix in PREFIX.items():
    files = sorted([p for p in (DATA_DIR/category).rglob("*") if p.is_file() and p.suffix.lower() == ".pdf"], key=lambda x: str(x).lower())
    records.extend([{"inventory_id": f"{prefix}{i:03d}", "category": category, "filename": p.name, "extension": p.suffix.lower(), "relative_path": str(p.relative_to(PROJECT_ROOT)), "absolute_path": str(p.resolve())} for i, p in enumerate(files, 1)])

inventory_df = pd.DataFrame(records)

display(inventory_df)
print("TOTAL PDFs:", len(inventory_df))

,inventory_id,category,filename,extension,relative_path,absolute_path
0,T001,Textbooks,book1.pdf,.pdf,data\Textbooks\book1.pdf,C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Textbooks\book1.pdf
1,T002,Textbooks,book2.pdf,.pdf,data\Textbooks\book2.pdf,C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Textbooks\book2.pdf
2,T003,Textbooks,d2l-en.pdf,.pdf,data\Textbooks\d2l-en.pdf,C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Textbooks\d2l-en.pdf
3,T004,Textbooks,ESLII_print12_toc.pdf,.pdf,data\Textbooks\ESLII_print12_toc.pdf,C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Textbooks\ESLII_print12_toc.pdf
4,T005,Textbooks,mml-book.pdf,.pdf,data\Textbooks\mml-book.pdf,C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Textbooks\mml-book.pdf
5,T006,Textbooks,UnderstandingDeepLearning_02_09_26_C.pdf,.pdf,data\Textbooks\UnderstandingDeepLearning_02_09_26_C.pdf,C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Textbooks\UnderstandingDeepLearning_02_09_26_...
6,S001,Survey,survey_1.pdf,.pdf,data\Survey\survey_1.pdf,C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Survey\survey_1.pdf
7,S002,Survey,survey_10.pdf,.pdf,data\Survey\survey_10.pdf,C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Survey\survey_10.pdf
8,S003,Survey,survey_11.pdf,.pdf,data\Survey\survey_11.pdf,C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Survey\survey_11.pdf
9,S004,Survey,survey_12.pdf,.pdf,data\Survey\survey_12.pdf,C:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\data\Survey\survey_12.pdf


TOTAL PDFs: 53


## 1.3 Category Count Validation

In [5]:
actual_counts = inventory_df.groupby("category").size().to_dict()

count_df = pd.DataFrame([{"category": c, "expected": e, "actual": actual_counts.get(c, 0), "difference": actual_counts.get(c, 0)-e, "status": "PASS" if actual_counts.get(c, 0) == e else "MISMATCH"} for c, e in EXPECTED.items()])

total_row = pd.DataFrame([{"category": "TOTAL", "expected": EXPECTED_TOTAL, "actual": len(inventory_df), "difference": len(inventory_df)-EXPECTED_TOTAL, "status": "PASS" if len(inventory_df) == EXPECTED_TOTAL else "MISMATCH"}])
count_df = pd.concat([count_df, total_row], ignore_index=True)

display(count_df)

for _, r in count_df.iterrows(): print(("✅" if r["status"] == "PASS" else "❌"), f"{r['category']:<12}", f"{r['actual']} / {r['expected']}")

count_status = bool((count_df["status"] == "PASS").all())
print("\nSTATUS:", "PASS ✅" if count_status else "REVIEW REQUIRED ⚠️")

,category,expected,actual,difference,status
0,Textbooks,8,6,-2,MISMATCH
1,Survey,23,23,0,PASS
2,Landmark,24,24,0,PASS
3,TOTAL,55,53,-2,MISMATCH


❌ Textbooks    6 / 8
✅ Survey       23 / 23
✅ Landmark     24 / 24
❌ TOTAL        53 / 55

STATUS: REVIEW REQUIRED ⚠️


## 1.4 Unexpected File Detection

In [6]:
all_paths = [(c, p) for c in EXPECTED for p in (DATA_DIR/c).rglob("*")]

all_files_df = pd.DataFrame([{"category": c, "filename": p.name, "extension": p.suffix.lower(), "relative_path": str(p.relative_to(PROJECT_ROOT)), "is_pdf": p.suffix.lower() == ".pdf", "is_hidden": any(x.startswith(".") for x in p.relative_to(DATA_DIR/c).parts), "is_temp": p.suffix.lower() in {".tmp", ".part", ".crdownload", ".download"}} for c, p in all_paths if p.is_file()])

nested_dirs_df = pd.DataFrame([{"category": c, "directory": p.name, "relative_path": str(p.relative_to(PROJECT_ROOT))} for c, p in all_paths if p.is_dir() and p != DATA_DIR/c])

unexpected_files_df = all_files_df[~all_files_df["is_pdf"]].copy()
hidden_files_df = all_files_df[all_files_df["is_hidden"]].copy()
temp_files_df = all_files_df[all_files_df["is_temp"]].copy()

summary_df = pd.DataFrame({
    "check": ["Non-PDF Files", "Hidden Files", "Temporary/Incomplete Files", "Nested Directories"],
    "count": [len(unexpected_files_df), len(hidden_files_df), len(temp_files_df), len(nested_dirs_df)]
})

display(summary_df)

if len(unexpected_files_df): display(unexpected_files_df)
if len(hidden_files_df): display(hidden_files_df)
if len(temp_files_df): display(temp_files_df)
if len(nested_dirs_df): display(nested_dirs_df)

unexpected_status = len(unexpected_files_df) == 0 and len(temp_files_df) == 0
print("STATUS:", "PASS ✅" if unexpected_status else "REVIEW REQUIRED ⚠️")

,check,count
0,Non-PDF Files,0
1,Hidden Files,0
2,Temporary/Incomplete Files,0
3,Nested Directories,0


STATUS: PASS ✅


## 1.5 Section 1 Summary

In [7]:
section_1_summary = pd.DataFrame([
    {"check": "Project Folders", "result": "PASS" if folder_status else "FAIL"},
    {"check": "Expected Document Counts", "result": "PASS" if count_status else "REVIEW"},
    {"check": "Unexpected Files", "result": "PASS" if unexpected_status else "REVIEW"},
    {"check": "Expected PDFs", "result": EXPECTED_TOTAL},
    {"check": "Discovered PDFs", "result": len(inventory_df)},
    {"check": "Textbooks", "result": f"{actual_counts.get('Textbooks', 0)}/8"},
    {"check": "Survey", "result": f"{actual_counts.get('Survey', 0)}/23"},
    {"check": "Landmark", "result": f"{actual_counts.get('Landmark', 0)}/24"},
])

display(section_1_summary)

final_status = folder_status and count_status and unexpected_status

print("="*55)
print("SECTION 1 STATUS:", "PASS ✅" if final_status else "REVIEW REQUIRED ⚠️")
print("="*55)

,check,result
0,Project Folders,PASS
1,Expected Document Counts,REVIEW
2,Unexpected Files,PASS
3,Expected PDFs,55
4,Discovered PDFs,53
5,Textbooks,6/8
6,Survey,23/23
7,Landmark,24/24


SECTION 1 STATUS: REVIEW REQUIRED ⚠️


In [8]:
METADATA_DIR = PROJECT_ROOT/"metadata"; METADATA_DIR.mkdir(exist_ok=True)

inventory_df.to_csv(METADATA_DIR/"01_file_inventory.csv", index=False, encoding="utf-8-sig")
count_df.to_csv(METADATA_DIR/"01_category_counts.csv", index=False, encoding="utf-8-sig")

print("Saved:", METADATA_DIR/"01_file_inventory.csv")
print("Saved:", METADATA_DIR/"01_category_counts.csv")

Saved: c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\01_file_inventory.csv
Saved: c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\01_category_counts.csv


# 2. PDF Integrity & File Health

This section validates the physical integrity of every PDF before document-level identity and content analysis.

### Checks
- PDF signature validation
- File open / corruption validation
- Encryption / password detection
- File size audit
- Page count audit

In [9]:
from pathlib import Path

inventory_df["pdf_signature"] = inventory_df["absolute_path"].apply(lambda p: Path(p).read_bytes()[:5] == b"%PDF-")
inventory_df["signature_status"] = inventory_df["pdf_signature"].map({True: "PASS", False: "FAIL"})

display(inventory_df[["inventory_id", "category", "filename", "pdf_signature", "signature_status"]])
print("Valid PDF signatures:", inventory_df["pdf_signature"].sum(), "/", len(inventory_df))

,inventory_id,category,filename,pdf_signature,signature_status
0,T001,Textbooks,book1.pdf,True,PASS
1,T002,Textbooks,book2.pdf,True,PASS
2,T003,Textbooks,d2l-en.pdf,True,PASS
3,T004,Textbooks,ESLII_print12_toc.pdf,True,PASS
4,T005,Textbooks,mml-book.pdf,True,PASS
5,T006,Textbooks,UnderstandingDeepLearning_02_09_26_C.pdf,True,PASS
6,S001,Survey,survey_1.pdf,True,PASS
7,S002,Survey,survey_10.pdf,True,PASS
8,S003,Survey,survey_11.pdf,True,PASS
9,S004,Survey,survey_12.pdf,True,PASS


Valid PDF signatures: 53 / 53


In [10]:
import fitz

def check_pdf_open(path):
    try:
        doc = fitz.open(path); pages = doc.page_count; doc.close()
        return True, pages, ""
    except Exception as e: return False, None, str(e)

open_results = inventory_df["absolute_path"].apply(check_pdf_open)
inventory_df["pdf_openable"] = open_results.apply(lambda x: x[0])
inventory_df["detected_pages"] = open_results.apply(lambda x: x[1])
inventory_df["open_error"] = open_results.apply(lambda x: x[2])
inventory_df["open_status"] = inventory_df["pdf_openable"].map({True: "PASS", False: "FAIL"})

display(inventory_df[["inventory_id", "filename", "pdf_openable", "detected_pages", "open_status", "open_error"]])
print("Openable PDFs:", inventory_df["pdf_openable"].sum(), "/", len(inventory_df))

,inventory_id,filename,pdf_openable,detected_pages,open_status,open_error
0,T001,book1.pdf,True,860,PASS,
1,T002,book2.pdf,True,1370,PASS,
2,T003,d2l-en.pdf,True,1151,PASS,
3,T004,ESLII_print12_toc.pdf,True,764,PASS,
4,T005,mml-book.pdf,True,417,PASS,
5,T006,UnderstandingDeepLearning_02_09_26_C.pdf,True,541,PASS,
6,S001,survey_1.pdf,True,144,PASS,
7,S002,survey_10.pdf,True,18,PASS,
8,S003,survey_11.pdf,True,40,PASS,
9,S004,survey_12.pdf,True,30,PASS,


Openable PDFs: 53 / 53


In [11]:
def check_encryption(path):
    try:
        doc = fitz.open(path); encrypted = bool(doc.is_encrypted); password_required = bool(doc.needs_pass); doc.close()
        return encrypted, password_required
    except: return None, None

enc_results = inventory_df["absolute_path"].apply(check_encryption)
inventory_df["encrypted"] = enc_results.apply(lambda x: x[0])
inventory_df["password_required"] = enc_results.apply(lambda x: x[1])
inventory_df["encryption_status"] = inventory_df.apply(lambda r: "FAIL" if r["password_required"] else ("REVIEW" if r["encrypted"] else "PASS"), axis=1)

display(inventory_df[["inventory_id", "filename", "encrypted", "password_required", "encryption_status"]])
print("Password-protected PDFs:", int(inventory_df["password_required"].fillna(False).sum()))

,inventory_id,filename,encrypted,password_required,encryption_status
0,T001,book1.pdf,False,False,PASS
1,T002,book2.pdf,False,False,PASS
2,T003,d2l-en.pdf,False,False,PASS
3,T004,ESLII_print12_toc.pdf,False,False,PASS
4,T005,mml-book.pdf,False,False,PASS
5,T006,UnderstandingDeepLearning_02_09_26_C.pdf,False,False,PASS
6,S001,survey_1.pdf,False,False,PASS
7,S002,survey_10.pdf,False,False,PASS
8,S003,survey_11.pdf,False,False,PASS
9,S004,survey_12.pdf,False,False,PASS


Password-protected PDFs: 0


In [12]:
inventory_df["file_size_bytes"] = inventory_df["absolute_path"].apply(lambda p: Path(p).stat().st_size)
inventory_df["file_size_mb"] = (inventory_df["file_size_bytes"] / (1024**2)).round(2)
inventory_df["size_flag"] = inventory_df.apply(lambda r: "REVIEW" if r["file_size_mb"] < (0.20 if r["category"] == "Textbooks" else 0.05) else "PASS", axis=1)

display(inventory_df[["inventory_id", "category", "filename", "file_size_mb", "size_flag"]].sort_values("file_size_mb"))
print("Total corpus size:", round(inventory_df["file_size_mb"].sum(), 2), "MB")

,inventory_id,category,filename,file_size_mb,size_flag
44,L016,Landmark,Landmark_23.pdf,0.17,PASS
45,L017,Landmark,Landmark_24.pdf,0.22,PASS
48,L020,Landmark,Landmark_5.pdf,0.37,PASS
37,L009,Landmark,Landmark_17.pdf,0.46,PASS
34,L006,Landmark,Landmark_14.pdf,0.51,PASS
43,L015,Landmark,Landmark_22.pdf,0.56,PASS
38,L010,Landmark,Landmark_18.pdf,0.61,PASS
41,L013,Landmark,Landmark_20.pdf,0.63,PASS
40,L012,Landmark,Landmark_2.pdf,0.74,PASS
30,L002,Landmark,Landmark_10.pdf,0.78,PASS


Total corpus size: 534.25 MB


In [13]:
inventory_df["page_count"] = inventory_df["detected_pages"].astype("Int64")
inventory_df["page_count_flag"] = inventory_df.apply(lambda r: "REVIEW" if pd.isna(r["page_count"]) or r["page_count"] <= 0 or (r["category"] == "Textbooks" and r["page_count"] < 100) or (r["category"] != "Textbooks" and r["page_count"] < 4) else "PASS", axis=1)

display(inventory_df[["inventory_id", "category", "filename", "page_count", "page_count_flag"]].sort_values(["category", "page_count"]))

,inventory_id,category,filename,page_count,page_count_flag
33,L005,Landmark,Landmark_13.pdf,8,PASS
34,L006,Landmark,Landmark_14.pdf,9,PASS
37,L009,Landmark,Landmark_17.pdf,9,PASS
52,L024,Landmark,Landmark_9.pdf,9,PASS
31,L003,Landmark,Landmark_11.pdf,10,PASS
44,L016,Landmark,Landmark_23.pdf,11,PASS
30,L002,Landmark,Landmark_10.pdf,12,PASS
39,L011,Landmark,Landmark_19.pdf,12,PASS
45,L017,Landmark,Landmark_24.pdf,12,PASS
48,L020,Landmark,Landmark_5.pdf,13,PASS


In [14]:
inventory_df["integrity_issues"] = inventory_df.apply(lambda r: ", ".join([x for x, bad in [("INVALID_SIGNATURE", not r["pdf_signature"]), ("UNOPENABLE", not r["pdf_openable"]), ("PASSWORD_REQUIRED", r["password_required"] == True), ("SUSPICIOUS_SIZE", r["size_flag"] == "REVIEW"), ("SUSPICIOUS_PAGE_COUNT", r["page_count_flag"] == "REVIEW")] if bad]), axis=1)
inventory_df["integrity_status"] = inventory_df.apply(lambda r: "FAIL" if (not r["pdf_signature"] or not r["pdf_openable"] or r["password_required"] == True) else ("REVIEW" if r["integrity_issues"] else "PASS"), axis=1)

display(inventory_df[["inventory_id", "category", "filename", "file_size_mb", "page_count", "integrity_status", "integrity_issues"]])

print(inventory_df["integrity_status"].value_counts())

,inventory_id,category,filename,file_size_mb,page_count,integrity_status,integrity_issues
0,T001,Textbooks,book1.pdf,88.05,860,PASS,
1,T002,Textbooks,book2.pdf,144.41,1370,PASS,
2,T003,Textbooks,d2l-en.pdf,42.62,1151,PASS,
3,T004,Textbooks,ESLII_print12_toc.pdf,12.78,764,PASS,
4,T005,Textbooks,mml-book.pdf,16.75,417,PASS,
5,T006,Textbooks,UnderstandingDeepLearning_02_09_26_C.pdf,21.31,541,PASS,
6,S001,Survey,survey_1.pdf,5.58,144,PASS,
7,S002,Survey,survey_10.pdf,1.04,18,PASS,
8,S003,Survey,survey_11.pdf,2.98,40,PASS,
9,S004,Survey,survey_12.pdf,7.30,30,PASS,


integrity_status
PASS    53
Name: count, dtype: int64


In [15]:
section_2_summary = pd.DataFrame([
    {"metric": "Total PDFs", "value": len(inventory_df)},
    {"metric": "Valid Signatures", "value": int(inventory_df["pdf_signature"].sum())},
    {"metric": "Openable PDFs", "value": int(inventory_df["pdf_openable"].sum())},
    {"metric": "Password Required", "value": int(inventory_df["password_required"].fillna(False).sum())},
    {"metric": "Total Corpus Size (MB)", "value": round(inventory_df["file_size_mb"].sum(), 2)},
    {"metric": "PASS", "value": int((inventory_df["integrity_status"] == "PASS").sum())},
    {"metric": "REVIEW", "value": int((inventory_df["integrity_status"] == "REVIEW").sum())},
    {"metric": "FAIL", "value": int((inventory_df["integrity_status"] == "FAIL").sum())},
])

display(section_2_summary)
print("SECTION 2 STATUS:", "PASS ✅" if (inventory_df["integrity_status"] != "FAIL").all() else "FAIL ❌")

,metric,value
0,Total PDFs,53.00
1,Valid Signatures,53.00
2,Openable PDFs,53.00
3,Password Required,0.00
4,Total Corpus Size (MB),534.25
5,PASS,53.00
6,REVIEW,0.00
7,FAIL,0.00


SECTION 2 STATUS: PASS ✅


In [16]:
inventory_df.to_csv(METADATA_DIR/"02_pdf_integrity_audit.csv", index=False, encoding="utf-8-sig")
print("Saved:", METADATA_DIR/"02_pdf_integrity_audit.csv")

Saved: c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\02_pdf_integrity_audit.csv


# 3. Document Identity, Completeness & Provenance

This section verifies that every PDF represents the intended research document, checks whether the file appears complete, matches it against the planned corpus, and records canonical source identifiers where available.

### Checks
- PDF metadata extraction
- Visible title and author detection
- Expected corpus matching
- Full document / preview / TOC detection
- Source provenance and research identifiers

In [17]:
def extract_pdf_metadata(path):
    try:
        doc = fitz.open(path); m = doc.metadata or {}; doc.close()
        return {"pdf_title": (m.get("title") or "").strip(), "pdf_author": (m.get("author") or "").strip(), "pdf_subject": (m.get("subject") or "").strip(), "pdf_creator": (m.get("creator") or "").strip(), "pdf_producer": (m.get("producer") or "").strip(), "pdf_creation_date": (m.get("creationDate") or "").strip()}
    except Exception as e: return {"pdf_title": "", "pdf_author": "", "pdf_subject": "", "pdf_creator": "", "pdf_producer": "", "pdf_creation_date": "", "metadata_error": str(e)}

metadata_df = pd.DataFrame(inventory_df["absolute_path"].apply(extract_pdf_metadata).tolist())
inventory_df = pd.concat([inventory_df.reset_index(drop=True), metadata_df.reset_index(drop=True)], axis=1)

display(inventory_df[["inventory_id", "filename", "pdf_title", "pdf_author", "pdf_creator", "pdf_producer"]])

,inventory_id,filename,pdf_title,pdf_author,pdf_creator,pdf_producer
0,T001,book1.pdf,,,LaTeX with hyperref,pdfTeX-1.40.26
1,T002,book2.pdf,,,LaTeX with hyperref,pdfTeX-1.40.27
2,T003,d2l-en.pdf,,,LaTeX with hyperref,xdvipdfmx (20210609)
3,T004,ESLII_print12_toc.pdf,Untitled,,dvips(k) 5.993 Copyright 2013 Radical Eye Software,GPL Ghostscript 9.07; modified using iText 2.1.7 by 1T3XT
4,T005,mml-book.pdf,Mathematics for Machine Learning,"Marc Peter Deisenroth, A. Aldo Faisal, Cheng Soon Ong",LaTeX with hyperref,pdfTeX-1.40.23
5,T006,UnderstandingDeepLearning_02_09_26_C.pdf,,,LaTeX with hyperref,xdvipdfmx (20200315)
6,S001,survey_1.pdf,A Survey of Large Language Models,Wayne Xin Zhao; Kun Zhou; Junyi Li; Tianyi Tang; Xiaolei Wang; Yupeng Hou; Yingqian Min; Beichen Zhang; Junjie Zhang...,arXiv GenPDF (tex2pdf:d72aa01),pikepdf 8.15.1
7,S002,survey_10.pdf,,,LaTeX with hyperref,pdfTeX-1.40.25
8,S003,survey_11.pdf,A Survey of Transformers,,LaTeX with acmart 2021/04/16 v1.77 Typesetting articles for the Association for Computing Machinery and hyperref 202...,"pdfTeX, Version 3.14159265-2.6-1.40.21 (TeX Live 2020) kpathsea version 6.3.2"
9,S004,survey_12.pdf,,,LaTeX with hyperref,pdfTeX-1.40.21


In [18]:
def extract_visible_identity(path):
    try:
        doc = fitz.open(path); page = doc[0]; data = page.get_text("dict"); text = page.get_text("text"); doc.close()
        spans = [{"text": s["text"].strip(), "size": s["size"], "y": s["bbox"][1]} for b in data.get("blocks", []) for l in b.get("lines", []) for s in l.get("spans", []) if s["text"].strip()]
        if not spans: return "", "", text[:800].replace("\n", " ")
        max_size = max(s["size"] for s in spans); title_spans = sorted([s for s in spans if s["size"] >= max_size * 0.85], key=lambda x: x["y"]); title = " ".join(s["text"] for s in title_spans)[:500]
        lines = [x.strip() for x in text.splitlines() if x.strip()]; title_norm = title.lower(); author_candidates = [x for x in lines[:12] if x.lower() not in title_norm and len(x) < 250 and not x.lower().startswith(("abstract", "arxiv", "doi"))]
        return title, " | ".join(author_candidates[:3]), " ".join(lines[:20])[:1200]
    except: return "", "", ""

visible = inventory_df["absolute_path"].apply(extract_visible_identity)
inventory_df["visible_title"] = visible.apply(lambda x: x[0])
inventory_df["visible_author_guess"] = visible.apply(lambda x: x[1])
inventory_df["first_page_preview"] = visible.apply(lambda x: x[2])

display(inventory_df[["inventory_id", "filename", "pdf_title", "visible_title", "pdf_author", "visible_author_guess"]])

,inventory_id,filename,pdf_title,visible_title,pdf_author,visible_author_guess
0,T001,book1.pdf,,Probabilistic Machine Learning,,
1,T002,book2.pdf,,Probabilistic Machine Learning,,
2,T003,d2l-en.pdf,,Dive into Deep Learning,,"ASTON ZHANG, ZACHARY C. LIPTON, MU LI, AND ALEXANDER J. | SMOLA"
3,T004,ESLII_print12_toc.pdf,Untitled,The Elements of Statistical Learning,,Springer Series in Statistics | Trevor Hastie | Robert Tibshirani
4,T005,mml-book.pdf,Mathematics for Machine Learning,MATHEMATICS MACHINE LEARNING,"Marc Peter Deisenroth, A. Aldo Faisal, Cheng Soon Ong",MATHEMATICS FOR | Marc Peter Deisenroth | A. Aldo Faisal
5,T006,UnderstandingDeepLearning_02_09_26_C.pdf,,Understanding Deep Learning,,"Simon J.D. Prince | February 8, 2026 | The most recent version of this document can be found at http://udlbook.com."
6,S001,survey_1.pdf,A Survey of Large Language Models,A Survey of Large Language Models L,Wayne Xin Zhao; Kun Zhou; Junyi Li; Tianyi Tang; Xiaolei Wang; Yupeng Hou; Yingqian Min; Beichen Zhang; Junjie Zhang...,"1 | Wayne Xin Zhao, Kun Zhou*, Junyi Li*, Tianyi Tang, Xiaolei Wang, Yupeng Hou, Yingqian Min, Beichen | Zhang, Junj..."
7,S002,survey_10.pdf,,A Survey on Multimodal Large Language Models R,,"IEEE TRANSACTIONS ON PATTERN ANALYSIS AND MACHINE INTELLIGENCE | 1 | Shukang Yin*, Chaoyou Fu*†, Sirui Zhao*, Ke Li,"
8,S003,survey_11.pdf,A Survey of Transformers,arXiv:2106.04554v2 [cs.LG] 15 Jun 2021,,"A Survey of Transformers | TIANYANG LIN, YUXIN WANG, XIANGYANG LIU, and XIPENG QIU∗, School of Computer | Science, F..."
9,S004,survey_12.pdf,,Transformers in Vision: A Survey T,,"1 | Salman Khan, Muzammal Naseer, Munawar Hayat, Syed Waqas Zamir, | Fahad Shahbaz Khan, and Mubarak Shah"


In [19]:
from difflib import SequenceMatcher
import re

EXPECTED_TITLES = {
    "Textbooks": [
        "Probabilistic Machine Learning: An Introduction",
        "Probabilistic Machine Learning: Advanced Topics",
        "Mathematics for Machine Learning",
        "The Elements of Statistical Learning: Data Mining, Inference, and Prediction",
        "Understanding Deep Learning",
        "Dive into Deep Learning"
    ],
    "Survey": [
        "A Survey of Large Language Models",
        "On the Opportunities and Risks of Foundation Models",
        "Retrieval-Augmented Generation for Large Language Models: A Survey",
        "A Survey on Evaluation of Large Language Models",
        "A Survey on Hallucination in Large Language Models",
        "The Rise and Potential of Large Language Model Based Agents: A Survey",
        "A Survey on In-context Learning",
        "Parameter-Efficient Fine-Tuning for Large Models: A Comprehensive Survey",
        "A Survey on Model Compression for Large Language Models",
        "A Survey on Multimodal Large Language Models",
        "A Survey of Transformers",
        "Transformers in Vision: A Survey",
        "Diffusion Models: A Comprehensive Survey of Methods and Applications",
        "Self-supervised Learning: Generative or Contrastive",
        "Graph Neural Networks: A Review of Methods and Applications",
        "A Comprehensive Survey on Transfer Learning",
        "AutoML: A Survey of the State-of-the-Art",
        "A Continual Learning Survey: Defying Forgetting in Classification Tasks",
        "Deep Learning for Generic Object Detection: A Survey",
        "Deep Learning for Anomaly Detection: A Review",
        "Explainable Artificial Intelligence (XAI): Concepts, Taxonomies, Opportunities and Challenges toward Responsible AI",
        "Deep Reinforcement Learning: A Brief Survey",
        "Geometric Deep Learning: Grids, Groups, Graphs, Geodesics, and Gauges"
    ],
    "Landmark": [
        "Attention Is All You Need",
        "BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding",
        "Language Models are Few-Shot Learners",
        "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "Dense Passage Retrieval for Open-Domain Question Answering",
        "LoRA: Low-Rank Adaptation of Large Language Models",
        "An Image Is Worth 16x16 Words: Transformers for Image Recognition at Scale",
        "Learning Transferable Visual Models From Natural Language Supervision",
        "ImageNet Classification with Deep Convolutional Neural Networks",
        "Deep Residual Learning for Image Recognition",
        "You Only Look Once: Unified, Real-Time Object Detection",
        "Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks",
        "U-Net: Convolutional Networks for Biomedical Image Segmentation",
        "Generative Adversarial Nets",
        "Auto-Encoding Variational Bayes",
        "Denoising Diffusion Probabilistic Models",
        "Playing Atari with Deep Reinforcement Learning",
        "Mastering Chess and Shogi by Self-Play with a General Reinforcement Learning Algorithm",
        "Proximal Policy Optimization Algorithms",
        "Continuous Control with Deep Reinforcement Learning",
        "Semi-Supervised Classification with Graph Convolutional Networks",
        "Adam: A Method for Stochastic Optimization",
        "Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift",
        "Efficient Estimation of Word Representations in Vector Space"
    ]
}

def norm_title(x): return re.sub(r"[^a-z0-9]+", " ", str(x).lower()).strip()

def match_expected(row):
    candidates = [x for x in [row["visible_title"], row["pdf_title"]] if isinstance(x, str) and len(x.strip()) > 3]
    scores = [(SequenceMatcher(None, norm_title(c), norm_title(e)).ratio(), e, c) for c in candidates for e in EXPECTED_TITLES[row["category"]]]
    if not scores: return "", "", 0.0, "UNKNOWN"
    score, expected, detected = max(scores, key=lambda x: x[0]); status = "MATCH" if score >= 0.82 else ("REVIEW" if score >= 0.65 else "MISMATCH")
    return expected, detected, round(score, 3), status

matches = inventory_df.apply(match_expected, axis=1)
inventory_df["expected_title"] = matches.apply(lambda x: x[0])
inventory_df["detected_title"] = matches.apply(lambda x: x[1])
inventory_df["title_match_score"] = matches.apply(lambda x: x[2])
inventory_df["title_match_status"] = matches.apply(lambda x: x[3])

display(inventory_df[["inventory_id", "category", "filename", "detected_title", "expected_title", "title_match_score", "title_match_status"]].sort_values("title_match_score"))
print(inventory_df["title_match_status"].value_counts())

,inventory_id,category,filename,detected_title,expected_title,title_match_score,title_match_status
22,S017,Survey,survey_3.pdf,,,0.000,UNKNOWN
31,L003,Landmark,Landmark_11.pdf,arXiv:1506.02640v5 [cs.CV] 9 May 2016,An Image Is Worth 16x16 Words: Transformers for Image Recognition at Scale,0.241,MISMATCH
42,L014,Landmark,Landmark_21.pdf,S -S C G C N arXiv:1609.02907v4 [cs.LG] 22 Feb 2017,Playing Atari with Deep Reinforcement Learning,0.255,MISMATCH
26,S021,Survey,survey_7.pdf,arXiv:2301.00234v6 [cs.CL] 5 Oct 2024,Deep Learning for Generic Object Detection: A Survey,0.256,MISMATCH
39,L011,Landmark,Landmark_19.pdf,arXiv:1707.06347v2 [cs.LG] 28 Aug 2017,Attention Is All You Need,0.262,MISMATCH
40,L012,Landmark,Landmark_2.pdf,arXiv:1810.04805v2 [cs.CL] 24 May 2019,Attention Is All You Need,0.262,MISMATCH
30,L002,Landmark,Landmark_10.pdf,arXiv:1512.03385v1 [cs.CV] 10 Dec 2015,Attention Is All You Need,0.262,MISMATCH
41,L013,Landmark,Landmark_20.pdf,C arXiv:1509.02971v6 [cs.LG] 5 Jul 2019,Playing Atari with Deep Reinforcement Learning,0.265,MISMATCH
15,S010,Survey,survey_19.pdf,arXiv:1809.02165v4 [cs.CV] 22 Aug 2019,Deep Learning for Generic Object Detection: A Survey,0.276,MISMATCH
23,S018,Survey,survey_4.pdf,arXiv:2307.03109v9 [cs.CL] 29 Dec 2023,Deep Learning for Generic Object Detection: A Survey,0.276,MISMATCH


title_match_status
MATCH       23
MISMATCH    16
REVIEW      13
UNKNOWN      1
Name: count, dtype: int64


In [20]:
def completeness_check(row):
    name = row["filename"].lower(); pages = row["page_count"] if not pd.isna(row["page_count"]) else 0
    if any(x in name for x in ["preview", "sample", "excerpt"]): return "PARTIAL_SUSPECTED", "filename indicates preview/sample"
    if any(x in name for x in ["_toc", "-toc", "toc.", "contents_only"]): return "TOC_SUSPECTED", "filename indicates TOC-only file"
    if row["category"] == "Textbooks" and pages < 100: return "REVIEW", f"textbook has only {pages} pages"
    if row["category"] != "Textbooks" and pages < 4: return "REVIEW", f"research paper has only {pages} pages"
    return "FULL_LIKELY", ""

completeness = inventory_df.apply(completeness_check, axis=1)
inventory_df["completeness_status"] = completeness.apply(lambda x: x[0])
inventory_df["completeness_reason"] = completeness.apply(lambda x: x[1])

display(inventory_df[["inventory_id", "category", "filename", "page_count", "completeness_status", "completeness_reason"]])
print(inventory_df["completeness_status"].value_counts())

,inventory_id,category,filename,page_count,completeness_status,completeness_reason
0,T001,Textbooks,book1.pdf,860,FULL_LIKELY,
1,T002,Textbooks,book2.pdf,1370,FULL_LIKELY,
2,T003,Textbooks,d2l-en.pdf,1151,FULL_LIKELY,
3,T004,Textbooks,ESLII_print12_toc.pdf,764,TOC_SUSPECTED,filename indicates TOC-only file
4,T005,Textbooks,mml-book.pdf,417,FULL_LIKELY,
5,T006,Textbooks,UnderstandingDeepLearning_02_09_26_C.pdf,541,FULL_LIKELY,
6,S001,Survey,survey_1.pdf,144,FULL_LIKELY,
7,S002,Survey,survey_10.pdf,18,FULL_LIKELY,
8,S003,Survey,survey_11.pdf,40,FULL_LIKELY,
9,S004,Survey,survey_12.pdf,30,FULL_LIKELY,


completeness_status
FULL_LIKELY      52
TOC_SUSPECTED     1
Name: count, dtype: int64


In [21]:
from datetime import datetime

def extract_identifiers(row):
    text = " ".join([str(row.get("pdf_subject", "")), str(row.get("first_page_preview", "")), str(row.get("pdf_title", ""))])
    arxiv = re.search(r"(?:arxiv[:\s]*)?(\d{4}\.\d{4,5})(?:v\d+)?", text, re.I); doi = re.search(r"\b10\.\d{4,9}/[-._;()/:A-Z0-9]+\b", text, re.I)
    path = Path(row["absolute_path"]); return arxiv.group(1) if arxiv else "", doi.group(0).rstrip(".,;)") if doi else "", datetime.fromtimestamp(path.stat().st_mtime).isoformat(timespec="seconds")

identifiers = inventory_df.apply(extract_identifiers, axis=1)
inventory_df["arxiv_id_candidate"] = identifiers.apply(lambda x: x[0])
inventory_df["doi_candidate"] = identifiers.apply(lambda x: x[1])
inventory_df["local_file_timestamp"] = identifiers.apply(lambda x: x[2])
inventory_df["provenance_type"] = inventory_df.apply(lambda r: "ARXIV_IDENTIFIED" if r["arxiv_id_candidate"] else ("DOI_IDENTIFIED" if r["doi_candidate"] else "LOCAL_SOURCE_ONLY"), axis=1)

display(inventory_df[["inventory_id", "filename", "arxiv_id_candidate", "doi_candidate", "provenance_type", "local_file_timestamp"]])

,inventory_id,filename,arxiv_id_candidate,doi_candidate,provenance_type,local_file_timestamp
0,T001,book1.pdf,,,LOCAL_SOURCE_ONLY,2026-09-19T14:03:37
1,T002,book2.pdf,,,LOCAL_SOURCE_ONLY,2026-09-19T14:04:05
2,T003,d2l-en.pdf,,,LOCAL_SOURCE_ONLY,2026-09-19T14:05:46
3,T004,ESLII_print12_toc.pdf,,,LOCAL_SOURCE_ONLY,2026-09-19T14:04:44
4,T005,mml-book.pdf,,,LOCAL_SOURCE_ONLY,2026-09-19T14:04:00
5,T006,UnderstandingDeepLearning_02_09_26_C.pdf,,,LOCAL_SOURCE_ONLY,2026-09-19T14:05:25
6,S001,survey_1.pdf,,,LOCAL_SOURCE_ONLY,2026-09-19T14:08:42
7,S002,survey_10.pdf,,,LOCAL_SOURCE_ONLY,2026-09-19T14:10:39
8,S003,survey_11.pdf,,,LOCAL_SOURCE_ONLY,2026-09-19T14:10:48
9,S004,survey_12.pdf,,,LOCAL_SOURCE_ONLY,2026-09-19T14:11:00


In [22]:
inventory_df["identity_status"] = inventory_df.apply(lambda r: "FAIL" if r["title_match_status"] == "MISMATCH" else ("REVIEW" if r["title_match_status"] in {"REVIEW", "UNKNOWN"} or r["completeness_status"] in {"PARTIAL_SUSPECTED", "TOC_SUSPECTED", "REVIEW"} else "PASS"), axis=1)
inventory_df["identity_issues"] = inventory_df.apply(lambda r: ", ".join([x for x, bad in [("TITLE", r["title_match_status"] != "MATCH"), ("COMPLETENESS", r["completeness_status"] != "FULL_LIKELY")] if bad]), axis=1)

display(inventory_df[["inventory_id", "category", "filename", "expected_title", "title_match_score", "completeness_status", "provenance_type", "identity_status", "identity_issues"]].sort_values(["identity_status", "title_match_score"]))

print(inventory_df["identity_status"].value_counts())
print("SECTION 3 STATUS:", "PASS ✅" if (inventory_df["identity_status"] == "PASS").all() else "REVIEW REQUIRED ⚠️")

,inventory_id,category,filename,expected_title,title_match_score,completeness_status,provenance_type,identity_status,identity_issues
31,L003,Landmark,Landmark_11.pdf,An Image Is Worth 16x16 Words: Transformers for Image Recognition at Scale,0.241,FULL_LIKELY,LOCAL_SOURCE_ONLY,FAIL,TITLE
42,L014,Landmark,Landmark_21.pdf,Playing Atari with Deep Reinforcement Learning,0.255,FULL_LIKELY,LOCAL_SOURCE_ONLY,FAIL,TITLE
26,S021,Survey,survey_7.pdf,Deep Learning for Generic Object Detection: A Survey,0.256,FULL_LIKELY,LOCAL_SOURCE_ONLY,FAIL,TITLE
30,L002,Landmark,Landmark_10.pdf,Attention Is All You Need,0.262,FULL_LIKELY,LOCAL_SOURCE_ONLY,FAIL,TITLE
39,L011,Landmark,Landmark_19.pdf,Attention Is All You Need,0.262,FULL_LIKELY,LOCAL_SOURCE_ONLY,FAIL,TITLE
40,L012,Landmark,Landmark_2.pdf,Attention Is All You Need,0.262,FULL_LIKELY,LOCAL_SOURCE_ONLY,FAIL,TITLE
41,L013,Landmark,Landmark_20.pdf,Playing Atari with Deep Reinforcement Learning,0.265,FULL_LIKELY,LOCAL_SOURCE_ONLY,FAIL,TITLE
15,S010,Survey,survey_19.pdf,Deep Learning for Generic Object Detection: A Survey,0.276,FULL_LIKELY,LOCAL_SOURCE_ONLY,FAIL,TITLE
19,S014,Survey,survey_21.pdf,Deep Learning for Generic Object Detection: A Survey,0.276,FULL_LIKELY,LOCAL_SOURCE_ONLY,FAIL,TITLE
23,S018,Survey,survey_4.pdf,Deep Learning for Generic Object Detection: A Survey,0.276,FULL_LIKELY,LOCAL_SOURCE_ONLY,FAIL,TITLE


identity_status
PASS      23
FAIL      16
REVIEW    14
Name: count, dtype: int64
SECTION 3 STATUS: REVIEW REQUIRED ⚠️


In [23]:
inventory_df.to_csv(METADATA_DIR/"03_document_identity_audit.csv", index=False, encoding="utf-8-sig")
print("Saved:", METADATA_DIR/"03_document_identity_audit.csv")

Saved: c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\03_document_identity_audit.csv


# 4. Structural & Extraction Preflight

This section performs lightweight structural checks before advanced parser experiments.

### Checks
- Basic text extractability
- Born-digital / scanned / mixed classification
- Page geometry and orientation
- Text-density anomalies
- Repeated header/footer detection
- Unicode and mathematical-symbol health

In [24]:
def text_extractability(path):
    try:
        doc = fitz.open(path); ids = sorted(set([0, doc.page_count//2, doc.page_count-1])); counts = [len(doc[i].get_text("text").strip()) for i in ids]; doc.close()
        return counts[0], counts[1] if len(counts) > 1 else counts[0], counts[-1], sum(c > 100 for c in counts)/len(counts)
    except: return 0, 0, 0, 0.0

text_check = inventory_df["absolute_path"].apply(text_extractability)
inventory_df["first_page_chars"] = text_check.apply(lambda x: x[0])
inventory_df["middle_page_chars"] = text_check.apply(lambda x: x[1])
inventory_df["last_page_chars"] = text_check.apply(lambda x: x[2])
inventory_df["sample_text_ratio"] = text_check.apply(lambda x: round(x[3], 2))
inventory_df["text_extractability"] = inventory_df["sample_text_ratio"].apply(lambda x: "PASS" if x >= 0.67 else ("REVIEW" if x > 0 else "NO_TEXT"))

display(inventory_df[["inventory_id", "filename", "first_page_chars", "middle_page_chars", "last_page_chars", "sample_text_ratio", "text_extractability"]])
print(inventory_df["text_extractability"].value_counts())

,inventory_id,filename,first_page_chars,middle_page_chars,last_page_chars,sample_text_ratio,text_extractability
0,T001,book1.pdf,30,1917,3569,0.67,PASS
1,T002,book2.pdf,30,2474,353,0.67,PASS
2,T003,d2l-en.pdf,85,1512,404,0.67,PASS
3,T004,ESLII_print12_toc.pdf,205,1873,1691,1.00,PASS
4,T005,mml-book.pdf,85,2487,827,0.67,PASS
5,T006,UnderstandingDeepLearning_02_09_26_C.pdf,683,1637,1176,1.00,PASS
6,S001,survey_1.pdf,6075,6799,2783,1.00,PASS
7,S002,survey_10.pdf,5819,6292,4550,1.00,PASS
8,S003,survey_11.pdf,3698,3288,3513,1.00,PASS
9,S004,survey_12.pdf,6048,5984,8285,1.00,PASS


text_extractability
PASS    53
Name: count, dtype: int64


In [25]:
import numpy as np

def classify_document_type(path):
    try:
        doc = fitz.open(path); ids = sorted(set(np.linspace(0, doc.page_count-1, min(7, doc.page_count), dtype=int))); chars = [len(doc[i].get_text("text").strip()) for i in ids]; images = [len(doc[i].get_images(full=True)) for i in ids]; doc.close()
        text_ratio = sum(c >= 200 for c in chars)/len(chars); image_ratio = sum(i > 0 for i in images)/len(images)
        doc_type = "BORN_DIGITAL" if text_ratio >= 0.8 else ("SCANNED_OR_IMAGE_HEAVY" if text_ratio <= 0.2 and image_ratio >= 0.5 else "MIXED")
        return len(ids), round(text_ratio, 2), round(image_ratio, 2), doc_type
    except: return 0, 0.0, 0.0, "UNKNOWN"

scan_check = inventory_df["absolute_path"].apply(classify_document_type)
inventory_df["structure_sample_pages"] = scan_check.apply(lambda x: x[0])
inventory_df["text_page_ratio"] = scan_check.apply(lambda x: x[1])
inventory_df["image_page_ratio"] = scan_check.apply(lambda x: x[2])
inventory_df["document_type"] = scan_check.apply(lambda x: x[3])

display(inventory_df[["inventory_id", "category", "filename", "text_page_ratio", "image_page_ratio", "document_type"]])
print(inventory_df["document_type"].value_counts())

,inventory_id,category,filename,text_page_ratio,image_page_ratio,document_type
0,T001,Textbooks,book1.pdf,0.0,0.0,UNKNOWN
1,T002,Textbooks,book2.pdf,0.0,0.0,UNKNOWN
2,T003,Textbooks,d2l-en.pdf,0.0,0.0,UNKNOWN
3,T004,Textbooks,ESLII_print12_toc.pdf,0.0,0.0,UNKNOWN
4,T005,Textbooks,mml-book.pdf,0.0,0.0,UNKNOWN
5,T006,Textbooks,UnderstandingDeepLearning_02_09_26_C.pdf,0.0,0.0,UNKNOWN
6,S001,Survey,survey_1.pdf,0.0,0.0,UNKNOWN
7,S002,Survey,survey_10.pdf,0.0,0.0,UNKNOWN
8,S003,Survey,survey_11.pdf,0.0,0.0,UNKNOWN
9,S004,Survey,survey_12.pdf,0.0,0.0,UNKNOWN


document_type
UNKNOWN    53
Name: count, dtype: int64


In [26]:
def geometry_profile(path):
    try:
        doc = fitz.open(path); dims = [(round(p.rect.width, 1), round(p.rect.height, 1)) for p in doc]; rotations = [p.rotation for p in doc]; portrait = sum(h >= w for w, h in dims); landscape = len(dims)-portrait; unique_dims = len(set(dims)); unique_rotations = sorted(set(rotations)); doc.close()
        return unique_dims, portrait, landscape, ",".join(map(str, unique_rotations)), unique_dims > 1 or len(unique_rotations) > 1
    except: return 0, 0, 0, "", True

geo = inventory_df["absolute_path"].apply(geometry_profile)
inventory_df["unique_page_sizes"] = geo.apply(lambda x: x[0])
inventory_df["portrait_pages"] = geo.apply(lambda x: x[1])
inventory_df["landscape_pages"] = geo.apply(lambda x: x[2])
inventory_df["page_rotations"] = geo.apply(lambda x: x[3])
inventory_df["mixed_geometry"] = geo.apply(lambda x: x[4])
inventory_df["geometry_status"] = inventory_df["mixed_geometry"].map({False: "PASS", True: "REVIEW"})

display(inventory_df[["inventory_id", "filename", "unique_page_sizes", "portrait_pages", "landscape_pages", "page_rotations", "geometry_status"]])

,inventory_id,filename,unique_page_sizes,portrait_pages,landscape_pages,page_rotations,geometry_status
0,T001,book1.pdf,1,860,0,0,PASS
1,T002,book2.pdf,1,1370,0,0,PASS
2,T003,d2l-en.pdf,1,1151,0,0,PASS
3,T004,ESLII_print12_toc.pdf,3,763,1,"0,90",REVIEW
4,T005,mml-book.pdf,2,417,0,0,REVIEW
5,T006,UnderstandingDeepLearning_02_09_26_C.pdf,1,541,0,0,PASS
6,S001,survey_1.pdf,1,144,0,0,PASS
7,S002,survey_10.pdf,1,18,0,0,PASS
8,S003,survey_11.pdf,1,40,0,0,PASS
9,S004,survey_12.pdf,1,30,0,0,PASS


In [27]:
def density_profile(path):
    try:
        doc = fitz.open(path); counts = np.array([len(p.get_text("text").strip()) for p in doc]); doc.close()
        median = float(np.median(counts)); threshold = max(30, median*0.05); zero = np.where(counts == 0)[0]+1; low = np.where((counts > 0) & (counts < threshold))[0]+1
        return round(median, 1), len(zero), len(low), ",".join(map(str, zero[:20])), ",".join(map(str, low[:20]))
    except: return 0.0, 0, 0, "", ""

density = inventory_df["absolute_path"].apply(density_profile)
inventory_df["median_chars_per_page"] = density.apply(lambda x: x[0])
inventory_df["zero_text_pages"] = density.apply(lambda x: x[1])
inventory_df["low_density_pages"] = density.apply(lambda x: x[2])
inventory_df["zero_text_page_samples"] = density.apply(lambda x: x[3])
inventory_df["low_density_page_samples"] = density.apply(lambda x: x[4])
inventory_df["density_status"] = inventory_df.apply(lambda r: "REVIEW" if r["zero_text_pages"] > max(3, r["page_count"]*0.05) else "PASS", axis=1)

display(inventory_df[["inventory_id", "filename", "page_count", "median_chars_per_page", "zero_text_pages", "low_density_pages", "density_status"]].sort_values("zero_text_pages", ascending=False))

,inventory_id,filename,page_count,median_chars_per_page,zero_text_pages,low_density_pages,density_status
1,T002,book2.pdf,1370,2385.0,28,9,PASS
0,T001,book1.pdf,860,2087.5,22,6,PASS
4,T005,mml-book.pdf,417,2165.0,6,1,PASS
5,T006,UnderstandingDeepLearning_02_09_26_C.pdf,541,2247.0,4,0,PASS
2,T003,d2l-en.pdf,1151,2000.0,2,12,PASS
21,S016,survey_23.pdf,160,2398.5,2,0,PASS
11,S006,survey_15.pdf,26,6410.5,1,0,PASS
7,S002,survey_10.pdf,18,6033.5,0,0,PASS
3,T004,ESLII_print12_toc.pdf,764,2034.0,0,11,PASS
8,S003,survey_11.pdf,40,3433.0,0,0,PASS


In [28]:
from collections import Counter

def repeated_headers_footers(path):
    try:
        doc = fitz.open(path); ids = sorted(set(np.linspace(0, doc.page_count-1, min(30, doc.page_count), dtype=int))); candidates = []
        for i in ids:
            page = doc[i]; h = page.rect.height
            for b in page.get_text("blocks"):
                text = re.sub(r"\s+", " ", b[4]).strip()
                if len(text) >= 4 and (b[1] < h*0.12 or b[3] > h*0.88) and not re.fullmatch(r"\d+", text): candidates.append(text)
        doc.close(); counts = Counter(candidates); repeated = [(t, n) for t, n in counts.items() if n >= max(3, len(ids)*0.35)]
        return len(repeated), " | ".join(f"{t} ({n})" for t, n in sorted(repeated, key=lambda x: -x[1])[:8])
    except: return 0, ""

header_footer = inventory_df["absolute_path"].apply(repeated_headers_footers)
inventory_df["repeated_header_footer_count"] = header_footer.apply(lambda x: x[0])
inventory_df["repeated_header_footer_samples"] = header_footer.apply(lambda x: x[1])
inventory_df["header_footer_status"] = inventory_df["repeated_header_footer_count"].apply(lambda x: "DETECTED" if x else "NONE")

display(inventory_df[["inventory_id", "filename", "repeated_header_footer_count", "repeated_header_footer_samples", "header_footer_status"]])

,inventory_id,filename,repeated_header_footer_count,repeated_header_footer_samples,header_footer_status
0,T001,book1.pdf,0,,NONE
1,T002,book2.pdf,0,,NONE
2,T003,d2l-en.pdf,0,,NONE
3,T004,ESLII_print12_toc.pdf,0,,NONE
4,T005,mml-book.pdf,0,,NONE
5,T006,UnderstandingDeepLearning_02_09_26_C.pdf,0,,NONE
6,S001,survey_1.pdf,0,,NONE
7,S002,survey_10.pdf,0,,NONE
8,S003,survey_11.pdf,0,,NONE
9,S004,survey_12.pdf,0,,NONE


In [29]:
MATH_SYMBOLS = "αβγδεζηθικλμνξοπρστυφχψωΓΔΘΛΞΠΣΦΨΩ∑∏∫√∇∂≤≥≈≠±×÷∞→←↔⊙∈∉⊂⊆∪∩"

def unicode_math_health(path):
    try:
        doc = fitz.open(path); ids = sorted(set(np.linspace(0, doc.page_count-1, min(15, doc.page_count), dtype=int))); text = "\n".join(doc[i].get_text("text") for i in ids); doc.close()
        replacements = text.count("\ufffd") + text.count("�") + text.count("□"); math_count = sum(text.count(s) for s in MATH_SYMBOLS); status = "REVIEW" if replacements > 5 else "PASS"
        return len(text), math_count, replacements, status
    except: return 0, 0, 0, "REVIEW"

unicode_check = inventory_df["absolute_path"].apply(unicode_math_health)
inventory_df["sampled_text_chars"] = unicode_check.apply(lambda x: x[0])
inventory_df["math_symbol_count"] = unicode_check.apply(lambda x: x[1])
inventory_df["replacement_symbol_count"] = unicode_check.apply(lambda x: x[2])
inventory_df["unicode_math_status"] = unicode_check.apply(lambda x: x[3])

display(inventory_df[["inventory_id", "filename", "sampled_text_chars", "math_symbol_count", "replacement_symbol_count", "unicode_math_status"]].sort_values("replacement_symbol_count", ascending=False))

,inventory_id,filename,sampled_text_chars,math_symbol_count,replacement_symbol_count,unicode_math_status
0,T001,book1.pdf,0,0,0,REVIEW
1,T002,book2.pdf,0,0,0,REVIEW
2,T003,d2l-en.pdf,0,0,0,REVIEW
3,T004,ESLII_print12_toc.pdf,0,0,0,REVIEW
4,T005,mml-book.pdf,0,0,0,REVIEW
5,T006,UnderstandingDeepLearning_02_09_26_C.pdf,0,0,0,REVIEW
6,S001,survey_1.pdf,0,0,0,REVIEW
7,S002,survey_10.pdf,0,0,0,REVIEW
8,S003,survey_11.pdf,0,0,0,REVIEW
9,S004,survey_12.pdf,0,0,0,REVIEW


In [30]:
inventory_df["structural_status"] = inventory_df.apply(lambda r: "REVIEW" if r["text_extractability"] != "PASS" or r["document_type"] == "UNKNOWN" or r["density_status"] == "REVIEW" or r["unicode_math_status"] == "REVIEW" else "PASS", axis=1)

section_4_summary = pd.DataFrame([
    {"metric": "Total Documents", "value": len(inventory_df)},
    {"metric": "Born Digital", "value": int((inventory_df["document_type"] == "BORN_DIGITAL").sum())},
    {"metric": "Mixed", "value": int((inventory_df["document_type"] == "MIXED").sum())},
    {"metric": "Scanned/Image Heavy", "value": int((inventory_df["document_type"] == "SCANNED_OR_IMAGE_HEAVY").sum())},
    {"metric": "Geometry Review", "value": int((inventory_df["geometry_status"] == "REVIEW").sum())},
    {"metric": "Density Review", "value": int((inventory_df["density_status"] == "REVIEW").sum())},
    {"metric": "Unicode/Math Review", "value": int((inventory_df["unicode_math_status"] == "REVIEW").sum())},
    {"metric": "Overall PASS", "value": int((inventory_df["structural_status"] == "PASS").sum())},
    {"metric": "Overall REVIEW", "value": int((inventory_df["structural_status"] == "REVIEW").sum())}
])

display(section_4_summary)
print("SECTION 4 STATUS:", "PASS ✅" if (inventory_df["structural_status"] == "PASS").all() else "REVIEW REQUIRED ⚠️")

,metric,value
0,Total Documents,53
1,Born Digital,0
2,Mixed,0
3,Scanned/Image Heavy,0
4,Geometry Review,2
5,Density Review,0
6,Unicode/Math Review,53
7,Overall PASS,0
8,Overall REVIEW,53


SECTION 4 STATUS: REVIEW REQUIRED ⚠️


In [31]:
inventory_df.to_csv(METADATA_DIR/"04_structural_preflight.csv", index=False, encoding="utf-8-sig")
print("Saved:", METADATA_DIR/"04_structural_preflight.csv")

Saved: c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\04_structural_preflight.csv


In [32]:
display(inventory_df[inventory_df["structural_status"] == "REVIEW"][["inventory_id", "filename", "document_type", "zero_text_pages", "mixed_geometry", "replacement_symbol_count", "structural_status"]])

,inventory_id,filename,document_type,zero_text_pages,mixed_geometry,replacement_symbol_count,structural_status
0,T001,book1.pdf,UNKNOWN,22,False,0,REVIEW
1,T002,book2.pdf,UNKNOWN,28,False,0,REVIEW
2,T003,d2l-en.pdf,UNKNOWN,2,False,0,REVIEW
3,T004,ESLII_print12_toc.pdf,UNKNOWN,0,True,0,REVIEW
4,T005,mml-book.pdf,UNKNOWN,6,True,0,REVIEW
5,T006,UnderstandingDeepLearning_02_09_26_C.pdf,UNKNOWN,4,False,0,REVIEW
6,S001,survey_1.pdf,UNKNOWN,0,False,0,REVIEW
7,S002,survey_10.pdf,UNKNOWN,0,False,0,REVIEW
8,S003,survey_11.pdf,UNKNOWN,0,False,0,REVIEW
9,S004,survey_12.pdf,UNKNOWN,0,False,0,REVIEW


In [33]:
def classify_document_type_v2(path):
    try:
        with fitz.open(path) as doc:
            ids = sorted(set(np.linspace(0, doc.page_count-1, min(7, doc.page_count), dtype=int).tolist()))
            chars, images = [], []
            for i in ids:
                page = doc.load_page(int(i)); chars.append(len(page.get_text("text").strip())); images.append(len(page.get_images(full=True)))
            text_ratio = sum(c >= 200 for c in chars)/len(chars); image_ratio = sum(i > 0 for i in images)/len(images)
            doc_type = "BORN_DIGITAL" if text_ratio >= 0.8 else ("SCANNED_OR_IMAGE_HEAVY" if text_ratio <= 0.2 and image_ratio >= 0.5 else "MIXED")
            return len(ids), round(text_ratio, 2), round(image_ratio, 2), doc_type
    except Exception as e: return 0, 0.0, 0.0, f"ERROR: {e}"

def unicode_math_health_v2(path):
    try:
        with fitz.open(path) as doc:
            ids = sorted(set(np.linspace(0, doc.page_count-1, min(15, doc.page_count), dtype=int).tolist()))
            text = "\n".join(doc.load_page(int(i)).get_text("text") for i in ids)
        replacements = text.count("\ufffd") + text.count("�") + text.count("□"); math_count = sum(text.count(s) for s in MATH_SYMBOLS)
        return len(text), math_count, replacements, "REVIEW" if replacements > 5 else "PASS"
    except Exception as e: return 0, 0, 0, f"ERROR: {e}"

scan_check = inventory_df["absolute_path"].apply(classify_document_type_v2)
inventory_df["structure_sample_pages"] = scan_check.apply(lambda x: x[0]); inventory_df["text_page_ratio"] = scan_check.apply(lambda x: x[1]); inventory_df["image_page_ratio"] = scan_check.apply(lambda x: x[2]); inventory_df["document_type"] = scan_check.apply(lambda x: x[3])

unicode_check = inventory_df["absolute_path"].apply(unicode_math_health_v2)
inventory_df["sampled_text_chars"] = unicode_check.apply(lambda x: x[0]); inventory_df["math_symbol_count"] = unicode_check.apply(lambda x: x[1]); inventory_df["replacement_symbol_count"] = unicode_check.apply(lambda x: x[2]); inventory_df["unicode_math_status"] = unicode_check.apply(lambda x: x[3])

inventory_df["structural_status"] = inventory_df.apply(lambda r: "REVIEW" if r["text_extractability"] != "PASS" or str(r["document_type"]).startswith("ERROR") or r["density_status"] == "REVIEW" or r["unicode_math_status"] != "PASS" else "PASS", axis=1)

display(pd.DataFrame({"document_type": inventory_df["document_type"].value_counts(), "count": inventory_df["document_type"].value_counts()}))
print("\nUnicode/Math:", inventory_df["unicode_math_status"].value_counts().to_dict())
print("Structural:", inventory_df["structural_status"].value_counts().to_dict())

,document_type,count
document_type,,
BORN_DIGITAL,52,52
MIXED,1,1



Unicode/Math: {'PASS': 53}
Structural: {'PASS': 53}


# 5. Navigation, Page Mapping & Duplicate Analysis

This section analyzes document navigation structures and detects exact or near-duplicate research sources.

### Checks
- Table-of-contents and navigation signals
- PDF bookmarks / document outline
- PDF page index vs printed page labels
- SHA-256 exact duplicate detection
- Near-duplicate content detection

In [34]:
import hashlib
from difflib import SequenceMatcher
from itertools import combinations

def navigation_profile(path):
    try:
        with fitz.open(path) as doc:
            toc = doc.get_toc(simple=True); labels = [doc.load_page(i).get_label() for i in range(doc.page_count)]; meaningful_labels = [(i+1, x) for i, x in enumerate(labels) if x and x != str(i+1)]
            return len(toc) > 0, len(toc), max([x[0] for x in toc], default=0), len(meaningful_labels), " | ".join(f"PDF {i}→{label}" for i, label in meaningful_labels[:10])
    except Exception as e: return False, 0, 0, 0, f"ERROR: {e}"

def sha256_file(path, chunk=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while data := f.read(chunk): h.update(data)
    return h.hexdigest()

def normalized_fingerprint_text(path, max_pages=12, max_chars=30000):
    try:
        with fitz.open(path) as doc:
            ids = sorted(set(np.linspace(0, doc.page_count-1, min(max_pages, doc.page_count), dtype=int).tolist()))
            text = " ".join(doc.load_page(int(i)).get_text("text") for i in ids)
        text = re.sub(r"\s+", " ", text.lower()); text = re.sub(r"[^a-z0-9 ]+", " ", text); return re.sub(r"\s+", " ", text).strip()[:max_chars]
    except: return ""

nav = inventory_df["absolute_path"].apply(navigation_profile)
inventory_df["has_outline"] = nav.apply(lambda x: x[0]); inventory_df["outline_entries"] = nav.apply(lambda x: x[1]); inventory_df["outline_depth"] = nav.apply(lambda x: x[2]); inventory_df["custom_page_labels"] = nav.apply(lambda x: x[3]); inventory_df["page_label_samples"] = nav.apply(lambda x: x[4])
inventory_df["toc_navigation_status"] = inventory_df.apply(lambda r: "AVAILABLE" if r["has_outline"] else "NOT_AVAILABLE", axis=1)

inventory_df["sha256"] = inventory_df["absolute_path"].apply(sha256_file)
hash_counts = inventory_df["sha256"].value_counts()
inventory_df["exact_duplicate"] = inventory_df["sha256"].map(hash_counts).gt(1)
inventory_df["exact_duplicate_group"] = inventory_df.apply(lambda r: r["sha256"][:12] if r["exact_duplicate"] else "", axis=1)

inventory_df["fingerprint_text"] = inventory_df["absolute_path"].apply(normalized_fingerprint_text)
near_duplicate_pairs = []
for i, j in combinations(inventory_df.index, 2):
    a, b = inventory_df.at[i, "fingerprint_text"], inventory_df.at[j, "fingerprint_text"]
    if not a or not b: continue
    title_a, title_b = str(inventory_df.at[i, "expected_title"]), str(inventory_df.at[j, "expected_title"])
    title_score = SequenceMatcher(None, norm_title(title_a), norm_title(title_b)).ratio() if title_a and title_b else 0
    content_score = SequenceMatcher(None, a[:12000], b[:12000]).ratio() if title_score >= 0.55 else 0
    if content_score >= 0.90: near_duplicate_pairs.append({"doc_a": inventory_df.at[i, "inventory_id"], "file_a": inventory_df.at[i, "filename"], "doc_b": inventory_df.at[j, "inventory_id"], "file_b": inventory_df.at[j, "filename"], "title_similarity": round(title_score, 3), "content_similarity": round(content_score, 3), "status": "PROBABLE_DUPLICATE"})

near_duplicate_df = pd.DataFrame(near_duplicate_pairs, columns=["doc_a", "file_a", "doc_b", "file_b", "title_similarity", "content_similarity", "status"])
near_ids = set(near_duplicate_df["doc_a"]).union(set(near_duplicate_df["doc_b"])) if len(near_duplicate_df) else set()
inventory_df["near_duplicate"] = inventory_df["inventory_id"].isin(near_ids)
inventory_df["duplicate_status"] = inventory_df.apply(lambda r: "EXACT_DUPLICATE" if r["exact_duplicate"] else ("REVIEW_NEAR_DUPLICATE" if r["near_duplicate"] else "UNIQUE"), axis=1)

inventory_df["navigation_duplicate_status"] = inventory_df.apply(lambda r: "REVIEW" if r["duplicate_status"] != "UNIQUE" else "PASS", axis=1)

summary_5 = pd.DataFrame([
    {"metric": "Documents", "value": len(inventory_df)},
    {"metric": "Documents with PDF Outline", "value": int(inventory_df["has_outline"].sum())},
    {"metric": "Documents with Custom Page Labels", "value": int((inventory_df["custom_page_labels"] > 0).sum())},
    {"metric": "Exact Duplicate Files", "value": int(inventory_df["exact_duplicate"].sum())},
    {"metric": "Near-Duplicate Files", "value": int(inventory_df["near_duplicate"].sum())},
    {"metric": "Unique Documents", "value": int((inventory_df["duplicate_status"] == "UNIQUE").sum())}
])

display(summary_5)
display(inventory_df[["inventory_id", "category", "filename", "has_outline", "outline_entries", "outline_depth", "custom_page_labels", "page_label_samples", "duplicate_status"]])

if len(near_duplicate_df): display(near_duplicate_df)
else: print("Near duplicates: NONE ✅")

exact_duplicates_df = inventory_df[inventory_df["exact_duplicate"]][["inventory_id", "category", "filename", "sha256", "exact_duplicate_group"]]
if len(exact_duplicates_df): display(exact_duplicates_df)
else: print("Exact duplicates: NONE ✅")

print("\nSECTION 5 STATUS:", "PASS ✅" if (inventory_df["duplicate_status"] == "UNIQUE").all() else "REVIEW REQUIRED ⚠️")

inventory_df.drop(columns=["fingerprint_text"], inplace=True)
inventory_df.to_csv(METADATA_DIR/"05_navigation_duplicate_audit.csv", index=False, encoding="utf-8-sig")
near_duplicate_df.to_csv(METADATA_DIR/"05_near_duplicate_report.csv", index=False, encoding="utf-8-sig")

print("Saved:", METADATA_DIR/"05_navigation_duplicate_audit.csv")
print("Saved:", METADATA_DIR/"05_near_duplicate_report.csv")

,metric,value
0,Documents,53
1,Documents with PDF Outline,40
2,Documents with Custom Page Labels,8
3,Exact Duplicate Files,0
4,Near-Duplicate Files,0
5,Unique Documents,53


,inventory_id,category,filename,has_outline,outline_entries,outline_depth,custom_page_labels,page_label_samples,duplicate_status
0,T001,Textbooks,book1.pdf,True,714,4,860,PDF 1→i | PDF 2→ii | PDF 3→iii | PDF 4→iv | PDF 5→v | PDF 6→vi | PDF 7→vii | PDF 8→viii | PDF 9→ix | PDF 10→x,UNIQUE
1,T002,Textbooks,book2.pdf,True,1071,4,1370,PDF 1→i | PDF 2→ii | PDF 3→iii | PDF 4→iv | PDF 5→v | PDF 6→vi | PDF 7→vii | PDF 8→viii | PDF 9→ix | PDF 10→x,UNIQUE
2,T003,Textbooks,d2l-en.pdf,True,1335,4,1151,PDF 1→i | PDF 2→ii | PDF 3→iii | PDF 4→iv | PDF 5→v | PDF 6→vi | PDF 7→vii | PDF 8→viii | PDF 9→ix | PDF 10→x,UNIQUE
3,T004,Textbooks,ESLII_print12_toc.pdf,True,375,8,0,,UNIQUE
4,T005,Textbooks,mml-book.pdf,True,104,3,415,PDF 3→i | PDF 4→ii | PDF 5→iii | PDF 6→iv | PDF 7→1 | PDF 8→2 | PDF 9→3 | PDF 10→4 | PDF 11→5 | PDF 12→6,UNIQUE
5,T006,Textbooks,UnderstandingDeepLearning_02_09_26_C.pdf,True,181,2,541,PDF 1→i | PDF 2→ii | PDF 3→i | PDF 4→ii | PDF 5→iii | PDF 6→iv | PDF 7→v | PDF 8→vi | PDF 9→vii | PDF 10→viii,UNIQUE
6,S001,Survey,survey_1.pdf,True,115,3,0,,UNIQUE
7,S002,Survey,survey_10.pdf,True,41,3,0,,UNIQUE
8,S003,Survey,survey_11.pdf,True,29,2,0,,UNIQUE
9,S004,Survey,survey_12.pdf,False,0,0,0,,UNIQUE


Near duplicates: NONE ✅
Exact duplicates: NONE ✅

SECTION 5 STATUS: PASS ✅
Saved: c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\05_navigation_duplicate_audit.csv
Saved: c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\05_near_duplicate_report.csv


# 6. Research Metadata & Corpus Analysis

This section establishes the canonical research identity of every document and analyzes the corpus as a whole.

### Checks
- Canonical work and version resolution
- DOI / arXiv / venue metadata consistency
- Primary research-topic coverage
- Source-type distribution
- Publication-year distribution
- Canonical identity conflicts

Automated title guesses from earlier sections are not treated as authoritative here. Canonical metadata is resolved against the intentionally curated RAGScholar corpus.

In [35]:
import re, hashlib
from pathlib import Path

CANONICAL = {
    "book1.pdf": ("Probabilistic Machine Learning: An Introduction", 2022, "General_ML"),
    "book2.pdf": ("Probabilistic Machine Learning: Advanced Topics", 2023, "General_ML"),
    "d2l-en.pdf": ("Dive into Deep Learning", 2023, "Deep_Learning"),
    "ESLII_print12_toc.pdf": ("The Elements of Statistical Learning: Data Mining, Inference, and Prediction", 2009, "General_ML"),
    "mml-book.pdf": ("Mathematics for Machine Learning", 2020, "General_ML"),
    "UnderstandingDeepLearning_02_09_26_C.pdf": ("Understanding Deep Learning", 2023, "Deep_Learning"),

    "survey_1.pdf": ("A Survey of Large Language Models", 2023, "NLP_LLM"),
    "survey_2.pdf": ("On the Opportunities and Risks of Foundation Models", 2021, "NLP_LLM"),
    "survey_3.pdf": ("Retrieval-Augmented Generation for Large Language Models: A Survey", 2023, "RAG_IR"),
    "survey_4.pdf": ("A Survey on Evaluation of Large Language Models", 2023, "NLP_LLM"),
    "survey_5.pdf": ("A Survey on Hallucination in Large Language Models", 2023, "NLP_LLM"),
    "survey_6.pdf": ("The Rise and Potential of Large Language Model Based Agents: A Survey", 2023, "NLP_LLM"),
    "survey_7.pdf": ("A Survey on In-context Learning", 2023, "NLP_LLM"),
    "survey_8.pdf": ("Parameter-Efficient Fine-Tuning for Large Models: A Comprehensive Survey", 2024, "NLP_LLM"),
    "survey_9.pdf": ("A Survey on Model Compression for Large Language Models", 2024, "NLP_LLM"),
    "survey_10.pdf": ("A Survey on Multimodal Large Language Models", 2023, "NLP_LLM"),
    "survey_11.pdf": ("A Survey of Transformers", 2021, "NLP_LLM"),
    "survey_12.pdf": ("Transformers in Vision: A Survey", 2021, "Computer_Vision"),
    "survey_13.pdf": ("Diffusion Models: A Comprehensive Survey of Methods and Applications", 2022, "Generative_AI"),
    "survey_1_14.pdf": ("Self-supervised Learning: Generative or Contrastive", 2020, "Deep_Learning"),
    "survey_15.pdf": ("Graph Neural Networks: A Review of Methods and Applications", 2018, "GNN"),
    "survey_16.pdf": ("A Comprehensive Survey on Transfer Learning", 2019, "General_ML"),
    "survey_17.pdf": ("AutoML: A Survey of the State-of-the-Art", 2019, "General_ML"),
    "survey_18.pdf": ("A Continual Learning Survey: Defying Forgetting in Classification Tasks", 2019, "Deep_Learning"),
    "survey_19.pdf": ("Deep Learning for Generic Object Detection: A Survey", 2018, "Computer_Vision"),
    "survey_20.pdf": ("Deep Learning for Anomaly Detection: A Review", 2020, "Deep_Learning"),
    "survey_21.pdf": ("Explainable Artificial Intelligence (XAI): Concepts, Taxonomies, Opportunities and Challenges toward Responsible AI", 2019, "General_ML"),
    "survey_22.pdf": ("Deep Reinforcement Learning: A Brief Survey", 2017, "Reinforcement_Learning"),
    "survey_23.pdf": ("Geometric Deep Learning: Grids, Groups, Graphs, Geodesics, and Gauges", 2021, "GNN"),

    "Landmark_1.pdf": ("Attention Is All You Need", 2017, "NLP_LLM"),
    "Landmark_2.pdf": ("BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding", 2018, "NLP_LLM"),
    "Landmark_3.pdf": ("Language Models are Few-Shot Learners", 2020, "NLP_LLM"),
    "Landmark_4.pdf": ("Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks", 2020, "RAG_IR"),
    "Landmark_5.pdf": ("Dense Passage Retrieval for Open-Domain Question Answering", 2020, "RAG_IR"),
    "Landmark_6.pdf": ("LoRA: Low-Rank Adaptation of Large Language Models", 2021, "NLP_LLM"),
    "Landmark_7.pdf": ("An Image Is Worth 16x16 Words: Transformers for Image Recognition at Scale", 2020, "Computer_Vision"),
    "Landmark_8.pdf": ("Learning Transferable Visual Models From Natural Language Supervision", 2021, "Computer_Vision"),
    "Landmark_9.pdf": ("ImageNet Classification with Deep Convolutional Neural Networks", 2012, "Computer_Vision"),
    "Landmark_10.pdf": ("Deep Residual Learning for Image Recognition", 2015, "Computer_Vision"),
    "Landmark_11.pdf": ("You Only Look Once: Unified, Real-Time Object Detection", 2015, "Computer_Vision"),
    "Landmark_12.pdf": ("Faster R-CNN: Towards Real-Time Object Detection with Region Proposal Networks", 2015, "Computer_Vision"),
    "Landmark_13.pdf": ("U-Net: Convolutional Networks for Biomedical Image Segmentation", 2015, "Computer_Vision"),
    "Landmark_14.pdf": ("Generative Adversarial Nets", 2014, "Generative_AI"),
    "Landmark_15.pdf": ("Auto-Encoding Variational Bayes", 2013, "Generative_AI"),
    "Landmark_16.pdf": ("Denoising Diffusion Probabilistic Models", 2020, "Generative_AI"),
    "Landmark_17.pdf": ("Playing Atari with Deep Reinforcement Learning", 2013, "Reinforcement_Learning"),
    "Landmark_18.pdf": ("Mastering Chess and Shogi by Self-Play with a General Reinforcement Learning Algorithm", 2017, "Reinforcement_Learning"),
    "Landmark_19.pdf": ("Proximal Policy Optimization Algorithms", 2017, "Reinforcement_Learning"),
    "Landmark_20.pdf": ("Continuous Control with Deep Reinforcement Learning", 2015, "Reinforcement_Learning"),
    "Landmark_21.pdf": ("Semi-Supervised Classification with Graph Convolutional Networks", 2016, "GNN"),
    "Landmark_22.pdf": ("Adam: A Method for Stochastic Optimization", 2014, "Deep_Learning"),
    "Landmark_23.pdf": ("Batch Normalization: Accelerating Deep Network Training by Reducing Internal Covariate Shift", 2015, "Deep_Learning"),
    "Landmark_24.pdf": ("Efficient Estimation of Word Representations in Vector Space", 2013, "NLP_LLM")
}

SOURCE_TYPE = {"Textbooks": "textbook", "Survey": "survey", "Landmark": "landmark"}

def slugify(x): return re.sub(r"[^a-z0-9]+", "_", x.lower()).strip("_")[:80]

def extract_venue(text):
    text = str(text)
    rules = [
        (r"\bNeurIPS\b|Neural Information Processing Systems", "NeurIPS"),
        (r"\bICLR\b|International Conference on Learning Representations", "ICLR"),
        (r"\bICML\b|International Conference on Machine Learning", "ICML"),
        (r"\bCVPR\b|Computer Vision and Pattern Recognition", "CVPR"),
        (r"\bEMNLP\b", "EMNLP"),
        (r"\bNAACL\b", "NAACL"),
        (r"\bACL\b", "ACL"),
        (r"\bJMLR\b|Journal of Machine Learning Research", "JMLR"),
        (r"IEEE Transactions on Pattern Analysis|TPAMI", "IEEE TPAMI"),
        (r"\bNature\b", "Nature"),
        (r"\bAI Open\b", "AI Open")
    ]
    return next((venue for pattern, venue in rules if re.search(pattern, text, re.I)), "")

def arxiv_year(arxiv_id):
    if not isinstance(arxiv_id, str) or not re.fullmatch(r"\d{4}\.\d{4,5}", arxiv_id.strip()): return None
    yy = int(arxiv_id[:2]); return 2000 + yy if yy <= 26 else 1900 + yy

inventory_df["canonical_title"] = inventory_df["filename"].map(lambda x: CANONICAL.get(x, ("", None, ""))[0])
inventory_df["publication_year"] = inventory_df["filename"].map(lambda x: CANONICAL.get(x, ("", None, ""))[1])
inventory_df["primary_topic"] = inventory_df["filename"].map(lambda x: CANONICAL.get(x, ("", None, ""))[2])
inventory_df["source_type"] = inventory_df["category"].map(SOURCE_TYPE)
inventory_df["canonical_work_id"] = inventory_df.apply(lambda r: f"{slugify(r['canonical_title'])}_{int(r['publication_year'])}" if r["canonical_title"] and pd.notna(r["publication_year"]) else "", axis=1)
inventory_df["version_id"] = inventory_df["sha256"].apply(lambda x: f"sha256_{str(x)[:12]}" if pd.notna(x) else "")
inventory_df["canonical_metadata_found"] = inventory_df["filename"].isin(CANONICAL)

combined_text = inventory_df.apply(lambda r: " ".join(str(r.get(c, "")) for c in ["pdf_title", "pdf_subject", "first_page_preview", "visible_title"]), axis=1)
inventory_df["venue_candidate"] = combined_text.apply(extract_venue)
inventory_df["arxiv_format_valid"] = inventory_df["arxiv_id_candidate"].fillna("").astype(str).apply(lambda x: bool(re.fullmatch(r"\d{4}\.\d{4,5}", x.strip())) if x.strip() else None)
inventory_df["doi_format_valid"] = inventory_df["doi_candidate"].fillna("").astype(str).apply(lambda x: bool(re.fullmatch(r"10\.\d{4,9}/\S+", x.strip(), re.I)) if x.strip() else None)
inventory_df["arxiv_year"] = inventory_df["arxiv_id_candidate"].apply(arxiv_year)
inventory_df["year_consistency"] = inventory_df.apply(lambda r: "PASS" if pd.isna(r["arxiv_year"]) or abs(int(r["publication_year"])-int(r["arxiv_year"])) <= 4 else "REVIEW", axis=1)

work_counts = inventory_df["canonical_work_id"].value_counts()
inventory_df["canonical_work_duplicate"] = inventory_df["canonical_work_id"].map(work_counts).gt(1)
inventory_df["canonical_status"] = inventory_df.apply(lambda r: "FAIL" if not r["canonical_metadata_found"] else ("REVIEW" if r["canonical_work_duplicate"] or r["year_consistency"] == "REVIEW" or r["arxiv_format_valid"] is False or r["doi_format_valid"] is False else "PASS"), axis=1)

topic_order = ["General_ML", "Deep_Learning", "NLP_LLM", "RAG_IR", "Computer_Vision", "Generative_AI", "GNN", "Reinforcement_Learning"]
topic_df = inventory_df.groupby("primary_topic").size().reindex(topic_order, fill_value=0).rename("documents").reset_index()
source_type_df = inventory_df.groupby("source_type").size().rename("documents").reset_index()
inventory_df["year_period"] = pd.cut(inventory_df["publication_year"], bins=[2000, 2014, 2019, 2022, 2026], labels=["≤2014", "2015–2019", "2020–2022", "2023–2026"], include_lowest=True)
year_df = inventory_df.groupby("year_period", observed=False).size().rename("documents").reset_index()

metadata_summary = pd.DataFrame([
    {"metric": "Total Documents", "value": len(inventory_df)},
    {"metric": "Canonical Metadata Resolved", "value": int(inventory_df["canonical_metadata_found"].sum())},
    {"metric": "Canonical Work-ID Conflicts", "value": int(inventory_df["canonical_work_duplicate"].sum())},
    {"metric": "Valid arXiv IDs Detected", "value": int((inventory_df["arxiv_format_valid"] == True).sum())},
    {"metric": "Valid DOIs Detected", "value": int((inventory_df["doi_format_valid"] == True).sum())},
    {"metric": "Venue Automatically Detected", "value": int(inventory_df["venue_candidate"].ne("").sum())},
    {"metric": "Year Consistency Reviews", "value": int((inventory_df["year_consistency"] == "REVIEW").sum())},
    {"metric": "Canonical PASS", "value": int((inventory_df["canonical_status"] == "PASS").sum())},
    {"metric": "Canonical REVIEW", "value": int((inventory_df["canonical_status"] == "REVIEW").sum())},
    {"metric": "Canonical FAIL", "value": int((inventory_df["canonical_status"] == "FAIL").sum())}
])

display(metadata_summary)
display(inventory_df[["inventory_id", "filename", "canonical_title", "publication_year", "source_type", "primary_topic", "arxiv_id_candidate", "doi_candidate", "venue_candidate", "canonical_work_id", "canonical_status"]])
display(topic_df)
display(source_type_df)
display(year_df)

review_df = inventory_df[inventory_df["canonical_status"] != "PASS"][["inventory_id", "filename", "canonical_title", "publication_year", "arxiv_id_candidate", "arxiv_year", "doi_candidate", "canonical_work_duplicate", "year_consistency", "canonical_status"]]
if len(review_df): display(review_df)
else: print("Canonical metadata issues: NONE ✅")

print("\nSECTION 6 STATUS:", "PASS ✅" if (inventory_df["canonical_status"] != "FAIL").all() and not inventory_df["canonical_work_duplicate"].any() else "REVIEW REQUIRED ⚠️")

inventory_df.to_csv(METADATA_DIR/"06_research_metadata_audit.csv", index=False, encoding="utf-8-sig")
topic_df.to_csv(METADATA_DIR/"06_topic_distribution.csv", index=False, encoding="utf-8-sig")
source_type_df.to_csv(METADATA_DIR/"06_source_type_distribution.csv", index=False, encoding="utf-8-sig")
year_df.to_csv(METADATA_DIR/"06_year_distribution.csv", index=False, encoding="utf-8-sig")

print("Saved:", METADATA_DIR/"06_research_metadata_audit.csv")
print("Saved:", METADATA_DIR/"06_topic_distribution.csv")
print("Saved:", METADATA_DIR/"06_source_type_distribution.csv")
print("Saved:", METADATA_DIR/"06_year_distribution.csv")

,metric,value
0,Total Documents,53
1,Canonical Metadata Resolved,53
2,Canonical Work-ID Conflicts,0
3,Valid arXiv IDs Detected,4
4,Valid DOIs Detected,2
5,Venue Automatically Detected,9
6,Year Consistency Reviews,0
7,Canonical PASS,53
8,Canonical REVIEW,0
9,Canonical FAIL,0


,inventory_id,filename,canonical_title,publication_year,source_type,primary_topic,arxiv_id_candidate,doi_candidate,venue_candidate,canonical_work_id,canonical_status
0,T001,book1.pdf,Probabilistic Machine Learning: An Introduction,2022,textbook,General_ML,,,,probabilistic_machine_learning_an_introduction_2022,PASS
1,T002,book2.pdf,Probabilistic Machine Learning: Advanced Topics,2023,textbook,General_ML,,,,probabilistic_machine_learning_advanced_topics_2023,PASS
2,T003,d2l-en.pdf,Dive into Deep Learning,2023,textbook,Deep_Learning,,,,dive_into_deep_learning_2023,PASS
3,T004,ESLII_print12_toc.pdf,"The Elements of Statistical Learning: Data Mining, Inference, and Prediction",2009,textbook,General_ML,,,,the_elements_of_statistical_learning_data_mining_inference_and_prediction_2009,PASS
4,T005,mml-book.pdf,Mathematics for Machine Learning,2020,textbook,General_ML,,,,mathematics_for_machine_learning_2020,PASS
5,T006,UnderstandingDeepLearning_02_09_26_C.pdf,Understanding Deep Learning,2023,textbook,Deep_Learning,,,,understanding_deep_learning_2023,PASS
6,S001,survey_1.pdf,A Survey of Large Language Models,2023,survey,NLP_LLM,,,,a_survey_of_large_language_models_2023,PASS
7,S002,survey_10.pdf,A Survey on Multimodal Large Language Models,2023,survey,NLP_LLM,,,IEEE TPAMI,a_survey_on_multimodal_large_language_models_2023,PASS
8,S003,survey_11.pdf,A Survey of Transformers,2021,survey,NLP_LLM,,,,a_survey_of_transformers_2021,PASS
9,S004,survey_12.pdf,Transformers in Vision: A Survey,2021,survey,Computer_Vision,,,,transformers_in_vision_a_survey_2021,PASS


,primary_topic,documents
0,General_ML,7
1,Deep_Learning,7
2,NLP_LLM,15
3,RAG_IR,3
4,Computer_Vision,9
5,Generative_AI,4
6,GNN,3
7,Reinforcement_Learning,5


,source_type,documents
0,landmark,24
1,survey,23
2,textbook,6


,year_period,documents
0,≤2014,7
1,2015–2019,18
2,2020–2022,16
3,2023–2026,12


Canonical metadata issues: NONE ✅

SECTION 6 STATUS: PASS ✅
Saved: c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\06_research_metadata_audit.csv
Saved: c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\06_topic_distribution.csv
Saved: c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\06_source_type_distribution.csv
Saved: c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\06_year_distribution.csv


# 7. Final Quality Gate & Canonical Registry

This section consolidates all previous corpus-audit results into the final research registry.

### Final Tasks
- Document-level quality scoring
- Hard-failure detection
- PASS / REVIEW / FAIL resolution
- Canonical corpus registry generation
- Audit-report export
- Final corpus summary
- Notebook 02 readiness gate

The intentionally curated corpus contains:

- Textbooks: 6
- Surveys: 23
- Landmark Papers: 24
- Total: 53

In [36]:
import json
from pathlib import Path

TARGET_CORPUS = {"Textbooks": 6, "Survey": 23, "Landmark": 24}
TARGET_TOTAL = sum(TARGET_CORPUS.values())

def col(name, default=None): return inventory_df[name] if name in inventory_df.columns else pd.Series([default]*len(inventory_df), index=inventory_df.index)

inventory_df["score_identity"] = inventory_df.apply(lambda r: 20 if r.get("canonical_metadata_found", False) else 0, axis=1)
inventory_df["score_completeness"] = inventory_df.apply(lambda r: 20 if pd.notna(r.get("page_count")) and ((r["category"] == "Textbooks" and r["page_count"] >= 100) or (r["category"] != "Textbooks" and r["page_count"] >= 4)) else 10 if pd.notna(r.get("page_count")) and r["page_count"] > 0 else 0, axis=1)
inventory_df["score_pdf_integrity"] = inventory_df.apply(lambda r: 15 if r.get("pdf_signature", False) and r.get("pdf_openable", False) and r.get("password_required", False) != True else 0, axis=1)
inventory_df["score_text_health"] = inventory_df.apply(lambda r: 15 if r.get("text_extractability") == "PASS" and r.get("density_status") == "PASS" and r.get("unicode_math_status") == "PASS" else 10 if r.get("text_extractability") == "PASS" else 0, axis=1)
inventory_df["score_structure"] = inventory_df.apply(lambda r: 10 if str(r.get("document_type", "")).startswith(("BORN_DIGITAL", "MIXED", "SCANNED_OR_IMAGE_HEAVY")) else 5 if r.get("pdf_openable", False) else 0, axis=1)
inventory_df["score_metadata"] = inventory_df.apply(lambda r: 10 if r.get("canonical_title") and pd.notna(r.get("publication_year")) and r.get("primary_topic") and r.get("source_type") else 5 if r.get("canonical_title") else 0, axis=1)
inventory_df["score_provenance"] = inventory_df.apply(lambda r: 10 if r.get("canonical_work_id") and r.get("version_id") and r.get("sha256") else 5 if r.get("sha256") else 0, axis=1)

inventory_df["quality_score"] = inventory_df[["score_identity", "score_completeness", "score_pdf_integrity", "score_text_health", "score_structure", "score_metadata", "score_provenance"]].sum(axis=1)

inventory_df["hard_failure_reasons"] = inventory_df.apply(lambda r: ", ".join([
    reason for reason, failed in [
        ("INVALID_PDF_SIGNATURE", r.get("pdf_signature", False) != True),
        ("PDF_UNOPENABLE", r.get("pdf_openable", False) != True),
        ("PASSWORD_REQUIRED", r.get("password_required", False) == True),
        ("CANONICAL_METADATA_MISSING", r.get("canonical_metadata_found", False) != True),
        ("EXACT_DUPLICATE", r.get("exact_duplicate", False) == True),
        ("CANONICAL_WORK_DUPLICATE", r.get("canonical_work_duplicate", False) == True)
    ] if failed
]), axis=1)

inventory_df["review_reasons"] = inventory_df.apply(lambda r: ", ".join([
    reason for reason, review in [
        ("LOW_TEXT_HEALTH", r.get("text_extractability") != "PASS" or r.get("density_status") != "PASS"),
        ("UNICODE_MATH_REVIEW", r.get("unicode_math_status") != "PASS"),
        ("STRUCTURE_UNKNOWN", str(r.get("document_type", "")).startswith(("UNKNOWN", "ERROR"))),
        ("PAGE_COUNT_REVIEW", r.get("page_count_flag") == "REVIEW"),
        ("NEAR_DUPLICATE", r.get("near_duplicate", False) == True),
        ("YEAR_REVIEW", r.get("year_consistency") == "REVIEW")
    ] if review
]), axis=1)

inventory_df["final_status"] = inventory_df.apply(lambda r: "FAIL" if r["hard_failure_reasons"] else ("REVIEW" if r["quality_score"] < 90 or r["review_reasons"] else "PASS"), axis=1)

inventory_df["quality_grade"] = pd.cut(inventory_df["quality_score"], bins=[-1, 74, 89, 100], labels=["FAIL", "REVIEW", "PASS"])

registry_columns = [
    "inventory_id", "canonical_work_id", "version_id", "category", "source_type", "filename", "canonical_title",
    "publication_year", "primary_topic", "pdf_author", "arxiv_id_candidate", "doi_candidate", "venue_candidate",
    "page_count", "file_size_mb", "sha256", "absolute_path", "relative_path", "document_type", "text_page_ratio",
    "image_page_ratio", "has_outline", "outline_entries", "outline_depth", "custom_page_labels", "page_label_samples",
    "exact_duplicate", "near_duplicate", "quality_score", "quality_grade", "final_status", "review_reasons", "hard_failure_reasons"
]

registry_columns = [c for c in registry_columns if c in inventory_df.columns]
corpus_registry = inventory_df[registry_columns].copy().sort_values(["category", "publication_year", "canonical_title"]).reset_index(drop=True)

actual_counts = inventory_df.groupby("category").size().to_dict()
missing_records = [{"category": c, "expected": e, "actual": actual_counts.get(c, 0), "missing": max(0, e-actual_counts.get(c, 0))} for c, e in TARGET_CORPUS.items() if actual_counts.get(c, 0) < e]
missing_documents_df = pd.DataFrame(missing_records, columns=["category", "expected", "actual", "missing"])

duplicate_report_df = inventory_df[(inventory_df["exact_duplicate"] == True) | (inventory_df["near_duplicate"] == True)][["inventory_id", "filename", "canonical_title", "exact_duplicate", "near_duplicate", "sha256"]].copy()

quality_report = inventory_df[["inventory_id", "category", "filename", "canonical_title", "score_identity", "score_completeness", "score_pdf_integrity", "score_text_health", "score_structure", "score_metadata", "score_provenance", "quality_score", "final_status", "review_reasons", "hard_failure_reasons"]].copy()

final_counts = inventory_df["final_status"].value_counts().to_dict()
critical_failures = int((inventory_df["final_status"] == "FAIL").sum())
review_count = int((inventory_df["final_status"] == "REVIEW").sum())
unique_count = int((inventory_df["duplicate_status"] == "UNIQUE").sum())
canonical_count = int(inventory_df["canonical_metadata_found"].sum())
valid_pdf_count = int((inventory_df["pdf_signature"] & inventory_df["pdf_openable"] & ~inventory_df["password_required"].fillna(False)).sum())

category_gate = all(actual_counts.get(c, 0) == e for c, e in TARGET_CORPUS.items())
duplicate_gate = not inventory_df["exact_duplicate"].any() and not inventory_df["canonical_work_duplicate"].any()
identity_gate = canonical_count == TARGET_TOTAL
pdf_gate = valid_pdf_count == TARGET_TOTAL
document_gate = len(inventory_df) == TARGET_TOTAL

notebook_02_ready = all([category_gate, duplicate_gate, identity_gate, pdf_gate, document_gate, critical_failures == 0])

corpus_summary = {
    "project": "RAGScholar",
    "target_documents": TARGET_TOTAL,
    "actual_documents": len(inventory_df),
    "categories": {c: {"expected": e, "actual": int(actual_counts.get(c, 0))} for c, e in TARGET_CORPUS.items()},
    "valid_pdfs": valid_pdf_count,
    "canonical_documents": canonical_count,
    "unique_documents": unique_count,
    "exact_duplicate_files": int(inventory_df["exact_duplicate"].sum()),
    "near_duplicate_files": int(inventory_df["near_duplicate"].sum()),
    "critical_failures": critical_failures,
    "review_documents": review_count,
    "pass_documents": int((inventory_df["final_status"] == "PASS").sum()),
    "mean_quality_score": round(float(inventory_df["quality_score"].mean()), 2),
    "minimum_quality_score": int(inventory_df["quality_score"].min()),
    "maximum_quality_score": int(inventory_df["quality_score"].max()),
    "notebook_02_ready": notebook_02_ready
}

display(quality_report.sort_values(["final_status", "quality_score"]))
display(pd.DataFrame([
    {"metric": "Target Documents", "value": TARGET_TOTAL},
    {"metric": "Actual Documents", "value": len(inventory_df)},
    {"metric": "Valid PDFs", "value": valid_pdf_count},
    {"metric": "Canonical Documents", "value": canonical_count},
    {"metric": "Unique Documents", "value": unique_count},
    {"metric": "PASS", "value": int((inventory_df["final_status"] == "PASS").sum())},
    {"metric": "REVIEW", "value": review_count},
    {"metric": "FAIL", "value": critical_failures},
    {"metric": "Mean Quality Score", "value": corpus_summary["mean_quality_score"]},
    {"metric": "Minimum Quality Score", "value": corpus_summary["minimum_quality_score"]}
]))

print("="*70)
print("RAGSCHOLAR — FINAL CORPUS AUDIT")
print("="*70)
print(f"Documents              : {len(inventory_df)} / {TARGET_TOTAL}")
print(f"Textbooks              : {actual_counts.get('Textbooks', 0)} / {TARGET_CORPUS['Textbooks']}")
print(f"Surveys                : {actual_counts.get('Survey', 0)} / {TARGET_CORPUS['Survey']}")
print(f"Landmark Papers        : {actual_counts.get('Landmark', 0)} / {TARGET_CORPUS['Landmark']}")
print("-"*70)
print(f"Valid PDFs             : {valid_pdf_count} / {TARGET_TOTAL}")
print(f"Canonical Documents    : {canonical_count} / {TARGET_TOTAL}")
print(f"Unique Documents       : {unique_count} / {TARGET_TOTAL}")
print(f"Exact Duplicates       : {int(inventory_df['exact_duplicate'].sum())}")
print(f"Near Duplicates        : {int(inventory_df['near_duplicate'].sum())}")
print("-"*70)
print(f"PASS                    : {int((inventory_df['final_status'] == 'PASS').sum())}")
print(f"REVIEW                  : {review_count}")
print(f"FAIL                    : {critical_failures}")
print(f"Mean Quality Score      : {corpus_summary['mean_quality_score']} / 100")
print("-"*70)
print("NOTEBOOK 02 GATE       :", "READY ✅" if notebook_02_ready else "NOT READY ❌")
print("="*70)

METADATA_DIR.mkdir(parents=True, exist_ok=True)
corpus_registry.to_csv(METADATA_DIR/"corpus_manifest.csv", index=False, encoding="utf-8-sig")
corpus_registry.to_json(METADATA_DIR/"corpus_manifest.jsonl", orient="records", lines=True, force_ascii=False)
quality_report.to_csv(METADATA_DIR/"corpus_audit_report.csv", index=False, encoding="utf-8-sig")
missing_documents_df.to_csv(METADATA_DIR/"missing_documents.csv", index=False, encoding="utf-8-sig")
duplicate_report_df.to_csv(METADATA_DIR/"duplicate_report.csv", index=False, encoding="utf-8-sig")

with open(METADATA_DIR/"corpus_sha256.json", "w", encoding="utf-8") as f: json.dump(dict(zip(inventory_df["canonical_work_id"], inventory_df["sha256"])), f, indent=2, ensure_ascii=False)

with open(METADATA_DIR/"corpus_summary.json", "w", encoding="utf-8") as f: json.dump(corpus_summary, f, indent=2, ensure_ascii=False)

print("\nFINAL FILES")
for name in ["corpus_manifest.csv", "corpus_manifest.jsonl", "corpus_sha256.json", "corpus_audit_report.csv", "missing_documents.csv", "duplicate_report.csv", "corpus_summary.json"]: print("✓", METADATA_DIR/name)

,inventory_id,category,filename,canonical_title,score_identity,score_completeness,score_pdf_integrity,score_text_health,score_structure,score_metadata,score_provenance,quality_score,final_status,review_reasons,hard_failure_reasons
0,T001,Textbooks,book1.pdf,Probabilistic Machine Learning: An Introduction,20,20,15,15,10,10,10,100,PASS,,
1,T002,Textbooks,book2.pdf,Probabilistic Machine Learning: Advanced Topics,20,20,15,15,10,10,10,100,PASS,,
2,T003,Textbooks,d2l-en.pdf,Dive into Deep Learning,20,20,15,15,10,10,10,100,PASS,,
3,T004,Textbooks,ESLII_print12_toc.pdf,"The Elements of Statistical Learning: Data Mining, Inference, and Prediction",20,20,15,15,10,10,10,100,PASS,,
4,T005,Textbooks,mml-book.pdf,Mathematics for Machine Learning,20,20,15,15,10,10,10,100,PASS,,
5,T006,Textbooks,UnderstandingDeepLearning_02_09_26_C.pdf,Understanding Deep Learning,20,20,15,15,10,10,10,100,PASS,,
6,S001,Survey,survey_1.pdf,A Survey of Large Language Models,20,20,15,15,10,10,10,100,PASS,,
7,S002,Survey,survey_10.pdf,A Survey on Multimodal Large Language Models,20,20,15,15,10,10,10,100,PASS,,
8,S003,Survey,survey_11.pdf,A Survey of Transformers,20,20,15,15,10,10,10,100,PASS,,
9,S004,Survey,survey_12.pdf,Transformers in Vision: A Survey,20,20,15,15,10,10,10,100,PASS,,


,metric,value
0,Target Documents,53.0
1,Actual Documents,53.0
2,Valid PDFs,53.0
3,Canonical Documents,53.0
4,Unique Documents,53.0
5,PASS,53.0
6,REVIEW,0.0
7,FAIL,0.0
8,Mean Quality Score,100.0
9,Minimum Quality Score,100.0


RAGSCHOLAR — FINAL CORPUS AUDIT
Documents              : 53 / 53
Textbooks              : 6 / 6
Surveys                : 23 / 23
Landmark Papers        : 24 / 24
----------------------------------------------------------------------
Valid PDFs             : 53 / 53
Canonical Documents    : 53 / 53
Unique Documents       : 53 / 53
Exact Duplicates       : 0
Near Duplicates        : 0
----------------------------------------------------------------------
PASS                    : 53
REVIEW                  : 0
FAIL                    : 0
Mean Quality Score      : 100.0 / 100
----------------------------------------------------------------------
NOTEBOOK 02 GATE       : READY ✅

FINAL FILES
✓ c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\corpus_manifest.csv
✓ c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\corpus_manifest.jsonl
✓ c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\corpus_sha256.json
✓ c:\Users\

# 8. Independent Final Corpus Certification

This is an independent end-of-notebook certification pass.

It re-validates the finalized corpus directly from disk instead of trusting previous notebook variables.

### Certification scope

- Final artifact consistency
- Expected corpus composition
- Physical file existence
- PDF signature and EOF validation
- Full-document page traversal
- Encryption and corruption checks
- Page-count consistency
- Text extraction health
- Page geometry sanity
- Unicode replacement-character anomalies
- SHA-256 recomputation
- Exact duplicate detection
- Independent near-duplicate screening
- Canonical ID uniqueness
- Metadata completeness
- Topic coverage
- Manifest CSV / JSONL consistency
- SHA registry consistency
- Final summary consistency
- Reproducible corpus snapshot fingerprint

A successful result certifies the source corpus for Notebook 02.

In [37]:
from pathlib import Path
import pandas as pd, numpy as np, fitz, hashlib, json, re, warnings
from itertools import combinations
from difflib import SequenceMatcher
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")

TARGET = {"Textbooks": 6, "Survey": 23, "Landmark": 24}
TARGET_TOTAL = sum(TARGET.values())
REQUIRED_TOPICS = {"General_ML", "Deep_Learning", "NLP_LLM", "RAG_IR", "Computer_Vision", "Generative_AI", "GNN", "Reinforcement_Learning"}
REQUIRED_ARTIFACTS = ["corpus_manifest.csv", "corpus_manifest.jsonl", "corpus_sha256.json", "corpus_audit_report.csv", "missing_documents.csv", "duplicate_report.csv", "corpus_summary.json"]

if "PROJECT_ROOT" not in globals():
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next((p for p in candidates if (p/"metadata"/"corpus_manifest.csv").exists()), None)
    assert PROJECT_ROOT is not None, "ScholarGpt project root could not be located."

PROJECT_ROOT = Path(PROJECT_ROOT)
METADATA_DIR = PROJECT_ROOT/"metadata"

checks, warnings_list, document_results = [], [], []

def add_check(name, passed, detail="", severity="CRITICAL"):
    checks.append({"check": name, "passed": bool(passed), "severity": severity, "detail": str(detail)})

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while chunk := f.read(1024*1024): h.update(chunk)
    return h.hexdigest()

def clean_text(x): return re.sub(r"\s+", " ", re.sub(r"[^a-z0-9 ]+", " ", str(x).lower())).strip()

artifact_state = {name: (METADATA_DIR/name).exists() for name in REQUIRED_ARTIFACTS}
add_check("All final audit artifacts exist", all(artifact_state.values()), [k for k, v in artifact_state.items() if not v])

manifest = pd.read_csv(METADATA_DIR/"corpus_manifest.csv")
manifest_jsonl = pd.read_json(METADATA_DIR/"corpus_manifest.jsonl", lines=True)

with open(METADATA_DIR/"corpus_sha256.json", encoding="utf-8") as f: sha_registry = json.load(f)
with open(METADATA_DIR/"corpus_summary.json", encoding="utf-8") as f: saved_summary = json.load(f)

try: missing_df = pd.read_csv(METADATA_DIR/"missing_documents.csv")
except: missing_df = pd.DataFrame()
try: duplicate_df = pd.read_csv(METADATA_DIR/"duplicate_report.csv")
except: duplicate_df = pd.DataFrame()

required_columns = {"inventory_id", "canonical_work_id", "version_id", "category", "source_type", "filename", "canonical_title", "publication_year", "primary_topic", "page_count", "sha256", "absolute_path", "relative_path", "quality_score", "final_status"}
add_check("Manifest schema complete", required_columns.issubset(manifest.columns), sorted(required_columns-set(manifest.columns)))
add_check("CSV manifest contains target document count", len(manifest) == TARGET_TOTAL, f"{len(manifest)} / {TARGET_TOTAL}")
add_check("JSONL manifest contains target document count", len(manifest_jsonl) == TARGET_TOTAL, f"{len(manifest_jsonl)} / {TARGET_TOTAL}")
add_check("CSV and JSONL canonical IDs identical", set(manifest["canonical_work_id"]) == set(manifest_jsonl["canonical_work_id"]), f"CSV={len(set(manifest['canonical_work_id']))}, JSONL={len(set(manifest_jsonl['canonical_work_id']))}")

category_counts = manifest["category"].value_counts().to_dict()
add_check("Category composition exact", all(category_counts.get(k, 0) == v for k, v in TARGET.items()), category_counts)
add_check("No missing corpus entries", missing_df.empty, f"{len(missing_df)} missing rows")
add_check("No existing duplicate-report entries", duplicate_df.empty, f"{len(duplicate_df)} rows")

add_check("Inventory IDs unique", manifest["inventory_id"].is_unique, f"unique={manifest['inventory_id'].nunique()}")
add_check("Canonical work IDs unique", manifest["canonical_work_id"].is_unique, f"unique={manifest['canonical_work_id'].nunique()}")
add_check("Version IDs unique", manifest["version_id"].is_unique, f"unique={manifest['version_id'].nunique()}")
add_check("Relative paths unique", manifest["relative_path"].is_unique, f"unique={manifest['relative_path'].nunique()}")
add_check("Canonical titles populated", manifest["canonical_title"].fillna("").str.strip().ne("").all())
add_check("Canonical IDs populated", manifest["canonical_work_id"].fillna("").str.strip().ne("").all())
add_check("SHA-256 populated", manifest["sha256"].fillna("").str.fullmatch(r"[0-9a-fA-F]{64}").all())
add_check("Publication years populated", pd.to_numeric(manifest["publication_year"], errors="coerce").notna().all())
add_check("Publication years plausible", pd.to_numeric(manifest["publication_year"], errors="coerce").between(1950, 2026).all())
add_check("All required research topics represented", REQUIRED_TOPICS.issubset(set(manifest["primary_topic"])), sorted(REQUIRED_TOPICS-set(manifest["primary_topic"])))

expected_source_type = {"Textbooks": "textbook", "Survey": "survey", "Landmark": "landmark"}
source_consistency = manifest.apply(lambda r: expected_source_type.get(r["category"]) == r["source_type"], axis=1)
add_check("Category/source-type mapping consistent", source_consistency.all(), f"{int((~source_consistency).sum())} mismatches")

for _, r in manifest.iterrows():
    path = Path(str(r["absolute_path"]))
    result = {"inventory_id": r["inventory_id"], "filename": r["filename"], "canonical_work_id": r["canonical_work_id"], "exists": path.exists(), "signature": False, "eof_marker": False, "openable": False, "encrypted": None, "password_required": None, "is_repaired": None, "manifest_pages": int(r["page_count"]) if pd.notna(r["page_count"]) else None, "actual_pages": None, "page_errors": 0, "invalid_geometry_pages": 0, "zero_text_pages": 0, "low_text_pages": 0, "total_chars": 0, "replacement_chars": 0, "manifest_sha256": str(r["sha256"]), "recomputed_sha256": "", "hash_match": False, "sample_text": ""}

    if not path.exists():
        document_results.append(result); continue

    try:
        with open(path, "rb") as f:
            result["signature"] = f.read(5) == b"%PDF-"
            f.seek(max(0, path.stat().st_size-8192)); result["eof_marker"] = b"%%EOF" in f.read()
    except: pass

    try:
        result["recomputed_sha256"] = file_sha256(path); result["hash_match"] = result["recomputed_sha256"].lower() == result["manifest_sha256"].lower()
    except: pass

    sample_fragments = []

    try:
        with fitz.open(path) as doc:
            result["openable"] = True; result["encrypted"] = bool(doc.is_encrypted); result["password_required"] = bool(doc.needs_pass); result["is_repaired"] = bool(getattr(doc, "is_repaired", False)); result["actual_pages"] = doc.page_count

            for i in range(doc.page_count):
                try:
                    page = doc.load_page(i); rect = page.rect
                    if rect.width <= 0 or rect.height <= 0: result["invalid_geometry_pages"] += 1
                    text = page.get_text("text") or ""; stripped = text.strip(); chars = len(stripped)
                    result["total_chars"] += chars
                    if chars == 0: result["zero_text_pages"] += 1
                    elif chars < 30: result["low_text_pages"] += 1
                    result["replacement_chars"] += text.count("\ufffd") + text.count("�") + text.count("□")
                    if i in {0, doc.page_count//4, doc.page_count//2, (3*doc.page_count)//4, doc.page_count-1}: sample_fragments.append(stripped[:5000])
                except: result["page_errors"] += 1
    except: pass

    result["sample_text"] = clean_text(" ".join(sample_fragments))[:20000]
    document_results.append(result)

validation_df = pd.DataFrame(document_results)

add_check("All physical PDFs exist", validation_df["exists"].all(), f"{int((~validation_df['exists']).sum())} missing")
add_check("All files have valid PDF signatures", validation_df["signature"].all(), f"{int((~validation_df['signature']).sum())} invalid")
add_check("All PDFs contain EOF markers", validation_df["eof_marker"].all(), f"{int((~validation_df['eof_marker']).sum())} without EOF marker")
add_check("All PDFs open successfully", validation_df["openable"].all(), f"{int((~validation_df['openable']).sum())} failed")
add_check("No PDF requires a password", ~validation_df["password_required"].fillna(True).any(), f"{int(validation_df['password_required'].fillna(True).sum())} protected")
add_check("No page-level extraction exceptions", validation_df["page_errors"].eq(0).all(), f"{int(validation_df['page_errors'].sum())} page errors")
add_check("No invalid page geometry", validation_df["invalid_geometry_pages"].eq(0).all(), f"{int(validation_df['invalid_geometry_pages'].sum())} invalid pages")
add_check("Manifest page counts match physical PDFs", validation_df.apply(lambda r: r["manifest_pages"] == r["actual_pages"], axis=1).all(), f"{int((validation_df['manifest_pages'] != validation_df['actual_pages']).sum())} mismatches")
add_check("All recomputed SHA-256 hashes match manifest", validation_df["hash_match"].all(), f"{int((~validation_df['hash_match']).sum())} mismatches")
add_check("All physical PDF hashes are unique", validation_df["recomputed_sha256"].nunique() == TARGET_TOTAL, f"unique={validation_df['recomputed_sha256'].nunique()}")
add_check("Every document yields extractable text", validation_df["total_chars"].gt(500).all(), f"{int((validation_df['total_chars'] <= 500).sum())} weak documents")
add_check("No excessive Unicode replacement corruption", validation_df["replacement_chars"].le(20).all(), f"max={int(validation_df['replacement_chars'].max())}", severity="WARNING")

blank_ratio = validation_df["zero_text_pages"] / validation_df["actual_pages"].replace(0, np.nan)
high_blank = blank_ratio > 0.10
add_check("No document has >10% zero-text pages", (~high_blank.fillna(True)).all(), f"{int(high_blank.sum())} documents", severity="WARNING")

repaired_docs = validation_df["is_repaired"].fillna(False)
add_check("No PDF required structural repair on open", (~repaired_docs).all(), f"{int(repaired_docs.sum())} repaired PDFs", severity="WARNING")

manifest_hash_map = dict(zip(manifest["canonical_work_id"], manifest["sha256"]))
add_check("SHA registry contains all canonical works", set(sha_registry.keys()) == set(manifest_hash_map.keys()), f"registry={len(sha_registry)}, manifest={len(manifest_hash_map)}")
add_check("SHA registry values match manifest", all(str(sha_registry.get(k, "")).lower() == str(v).lower() for k, v in manifest_hash_map.items()))

tfidf_texts = validation_df["sample_text"].fillna("").tolist()
near_pairs = []

if sum(bool(x.strip()) for x in tfidf_texts) >= 2:
    X = TfidfVectorizer(min_df=1, max_df=1.0, ngram_range=(1, 2), max_features=50000, sublinear_tf=True).fit_transform(tfidf_texts)
    sim = cosine_similarity(X)

    for i, j in combinations(range(len(manifest)), 2):
        if sim[i, j] >= 0.985:
            ta = clean_text(manifest.iloc[i]["canonical_title"]); tb = clean_text(manifest.iloc[j]["canonical_title"])
            title_similarity = SequenceMatcher(None, ta, tb).ratio()
            near_pairs.append({"doc_a": manifest.iloc[i]["inventory_id"], "doc_b": manifest.iloc[j]["inventory_id"], "file_a": manifest.iloc[i]["filename"], "file_b": manifest.iloc[j]["filename"], "content_similarity": round(float(sim[i, j]), 4), "title_similarity": round(title_similarity, 4)})

near_advanced_df = pd.DataFrame(near_pairs, columns=["doc_a", "doc_b", "file_a", "file_b", "content_similarity", "title_similarity"])
add_check("Independent near-duplicate screen clean", near_advanced_df.empty, f"{len(near_advanced_df)} suspicious pairs", severity="WARNING")

all_pass_status = manifest["final_status"].astype(str).str.upper().eq("PASS").all()
quality_scores = pd.to_numeric(manifest["quality_score"], errors="coerce")
add_check("Final manifest statuses are all PASS", all_pass_status, manifest["final_status"].value_counts().to_dict())
add_check("All document quality scores >= 90", quality_scores.ge(90).all(), f"min={quality_scores.min():.1f}")
add_check("No critical failure reasons remain", manifest.get("hard_failure_reasons", pd.Series([""]*len(manifest))).fillna("").astype(str).str.strip().eq("").all())
add_check("No review reasons remain", manifest.get("review_reasons", pd.Series([""]*len(manifest))).fillna("").astype(str).str.strip().eq("").all())

summary_expected = {
    "target_documents": TARGET_TOTAL,
    "actual_documents": TARGET_TOTAL,
    "valid_pdfs": TARGET_TOTAL,
    "canonical_documents": TARGET_TOTAL,
    "unique_documents": TARGET_TOTAL,
    "exact_duplicate_files": 0,
    "near_duplicate_files": 0,
    "critical_failures": 0,
    "review_documents": 0,
    "pass_documents": TARGET_TOTAL,
    "notebook_02_ready": True
}

summary_mismatches = {k: {"saved": saved_summary.get(k), "expected": v} for k, v in summary_expected.items() if saved_summary.get(k) != v}
add_check("Saved corpus_summary.json agrees with certification expectations", not summary_mismatches, summary_mismatches)

snapshot_payload = "\n".join(f"{r['canonical_work_id']}|{r['sha256']}" for _, r in manifest.sort_values("canonical_work_id").iterrows())
snapshot_sha256 = hashlib.sha256(snapshot_payload.encode("utf-8")).hexdigest()

checks_df = pd.DataFrame(checks)
critical_df = checks_df[checks_df["severity"] == "CRITICAL"]
warning_df = checks_df[checks_df["severity"] == "WARNING"]

critical_failures = critical_df[~critical_df["passed"]]
warning_failures = warning_df[~warning_df["passed"]]

if critical_failures.empty and warning_failures.empty: certification = "CERTIFIED ✅"
elif critical_failures.empty: certification = "CERTIFIED WITH NON-BLOCKING WARNINGS ⚠️"
else: certification = "NOT CERTIFIED ❌"

certification_report = {
    "project": "RAGScholar",
    "certification_scope": "Notebook 01 Source Corpus Audit",
    "target_documents": TARGET_TOTAL,
    "validated_documents": len(validation_df),
    "critical_checks": int(len(critical_df)),
    "critical_checks_passed": int(critical_df["passed"].sum()),
    "critical_failures": int(len(critical_failures)),
    "warning_checks": int(len(warning_df)),
    "warnings_triggered": int(len(warning_failures)),
    "physical_pdf_hashes_unique": int(validation_df["recomputed_sha256"].nunique()),
    "total_physical_pages": int(validation_df["actual_pages"].sum()),
    "total_extracted_characters": int(validation_df["total_chars"].sum()),
    "zero_text_pages": int(validation_df["zero_text_pages"].sum()),
    "page_extraction_errors": int(validation_df["page_errors"].sum()),
    "near_duplicate_candidates": int(len(near_advanced_df)),
    "corpus_snapshot_sha256": snapshot_sha256,
    "certification": certification,
    "ready_for_notebook_02": bool(critical_failures.empty)
}

display(checks_df.style.apply(lambda row: ["background-color: #d4edda" if row["passed"] else ("background-color: #fff3cd" if row["severity"] == "WARNING" else "background-color: #f8d7da")]*len(row), axis=1))
display(validation_df[["inventory_id", "filename", "actual_pages", "total_chars", "zero_text_pages", "low_text_pages", "replacement_chars", "is_repaired", "hash_match"]])

if len(near_advanced_df): display(near_advanced_df)

print("="*88)
print("RAGSCHOLAR — INDEPENDENT FINAL CORPUS CERTIFICATION")
print("="*88)
print(f"Documents validated          : {len(validation_df)} / {TARGET_TOTAL}")
print(f"Physical PDF pages           : {certification_report['total_physical_pages']:,}")
print(f"Extracted text characters    : {certification_report['total_extracted_characters']:,}")
print(f"Page extraction errors       : {certification_report['page_extraction_errors']}")
print(f"Unique SHA-256 files         : {certification_report['physical_pdf_hashes_unique']} / {TARGET_TOTAL}")
print(f"Near-duplicate candidates    : {certification_report['near_duplicate_candidates']}")
print("-"*88)
print(f"Critical checks              : {certification_report['critical_checks']}")
print(f"Critical checks passed       : {certification_report['critical_checks_passed']}")
print(f"Critical failures            : {certification_report['critical_failures']}")
print(f"Non-blocking warnings        : {certification_report['warnings_triggered']}")
print("-"*88)
print("CORPUS SNAPSHOT SHA-256      :", snapshot_sha256)
print("-"*88)
print("FINAL CERTIFICATION          :", certification)
print("NOTEBOOK 02                  :", "READY ✅" if critical_failures.empty else "NOT READY ❌")
print("="*88)

validation_df.drop(columns=["sample_text"], errors="ignore").to_csv(METADATA_DIR/"final_document_validation.csv", index=False, encoding="utf-8-sig")
checks_df.to_csv(METADATA_DIR/"final_certification_checks.csv", index=False, encoding="utf-8-sig")
near_advanced_df.to_csv(METADATA_DIR/"final_near_duplicate_screen.csv", index=False, encoding="utf-8-sig")

with open(METADATA_DIR/"final_certification_report.json", "w", encoding="utf-8") as f: json.dump(certification_report, f, indent=2, ensure_ascii=False)
with open(METADATA_DIR/"corpus_snapshot.sha256", "w", encoding="utf-8") as f: f.write(snapshot_sha256 + "\n")

print("\nCertification artifacts:")
for name in ["final_document_validation.csv", "final_certification_checks.csv", "final_near_duplicate_screen.csv", "final_certification_report.json", "corpus_snapshot.sha256"]: print("✓", METADATA_DIR/name)

,check,passed,severity,detail
0,All final audit artifacts exist,True,CRITICAL,[]
1,Manifest schema complete,True,CRITICAL,[]
2,CSV manifest contains target document count,True,CRITICAL,53 / 53
3,JSONL manifest contains target document count,True,CRITICAL,53 / 53
4,CSV and JSONL canonical IDs identical,True,CRITICAL,"CSV=53, JSONL=53"
5,Category composition exact,True,CRITICAL,"{'Landmark': 24, 'Survey': 23, 'Textbooks': 6}"
6,No missing corpus entries,True,CRITICAL,0 missing rows
7,No existing duplicate-report entries,True,CRITICAL,0 rows
8,Inventory IDs unique,True,CRITICAL,unique=53
9,Canonical work IDs unique,True,CRITICAL,unique=53


,inventory_id,filename,actual_pages,total_chars,zero_text_pages,low_text_pages,replacement_chars,is_repaired,hash_match
0,L024,Landmark_9.pdf,9,35323,0,0,0,False,True
1,L007,Landmark_15.pdf,14,41431,0,0,0,False,True
2,L017,Landmark_24.pdf,12,38732,0,0,0,False,True
3,L009,Landmark_17.pdf,9,36011,0,0,0,False,True
4,L015,Landmark_22.pdf,15,41889,0,0,0,False,True
5,L006,Landmark_14.pdf,9,29001,0,0,0,False,True
6,L016,Landmark_23.pdf,11,45499,0,0,0,False,True
7,L013,Landmark_20.pdf,14,47002,0,0,0,False,True
8,L002,Landmark_10.pdf,12,59413,0,0,0,False,True
9,L004,Landmark_12.pdf,14,65180,0,0,0,False,True


RAGSCHOLAR — INDEPENDENT FINAL CORPUS CERTIFICATION
Documents validated          : 53 / 53
Physical PDF pages           : 6,794
Extracted text characters    : 18,904,772
Page extraction errors       : 0
Unique SHA-256 files         : 53 / 53
Near-duplicate candidates    : 0
----------------------------------------------------------------------------------------
Critical checks              : 37
Critical checks passed       : 37
Critical failures            : 0
Non-blocking warnings        : 0
----------------------------------------------------------------------------------------
CORPUS SNAPSHOT SHA-256      : 09e5e430bcdcc874ebe4291fca74594e4fa91264723a0f54571bac8fdfc80194
----------------------------------------------------------------------------------------
FINAL CERTIFICATION          : CERTIFIED ✅
NOTEBOOK 02                  : READY ✅

Certification artifacts:
✓ c:\Users\USER\Kawsar_Ahmmed\ComeBack2026\Generative_Ai\ScholarGpt\metadata\final_document_validation.csv
✓ c:\Users\US